In [ ]:
# ── CAPTURE CELL : Run this FIRST before any other cell ─────────────────────
# All print() output appears in BOTH the cell output AND ps4_outputs/console.log
# All plt.show() figures appear in BOTH the cell output AND ps4_outputs/<name>.png
import os, sys
OUT_DIR = "ps4_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

class _Tee:
    """Writes to Jupyter cell output + log file simultaneously."""
    def __init__(self, jupyter_stream, log_fh):
        self._jupyter = jupyter_stream   # Jupyter OutStream -> shows in cell
        self._log     = log_fh           # file handle       -> saved to disk
        self._is_tee  = True
    def write(self, data):
        try:
            self._jupyter.write(data)
            self._jupyter.flush()
        except Exception:
            pass
        try:
            self._log.write(data)
            self._log.flush()
        except Exception:
            pass
    def flush(self):
        try: self._jupyter.flush()
        except Exception: pass
        try: self._log.flush()
        except Exception: pass
    def __getattr__(self, name):
        return getattr(self._jupyter, name)

if not getattr(sys.stdout, "_is_tee", False):
    _log_fh    = open(os.path.join(OUT_DIR, "console.log"), "w", buffering=1, encoding="utf-8")
    sys.stdout = _Tee(sys.stdout, _log_fh)
    sys.stderr = _Tee(sys.stderr, _log_fh)
    print("[capture] All output -> cell output + " + os.path.join(OUT_DIR, "console.log"))
else:
    print("[capture] Already active.")

try:
    import matplotlib.pyplot as _plt
    if not getattr(_plt.show, "_wrapped", False):
        _orig_show = _plt.show
        # Counter stored as mutable default arg on the function itself --
        # immune to any variable-name collision from other cells in the notebook.
        def _show(*a, _cnt=[0], **k):
            for _fig_num in _plt.get_fignums():
                _fig = _plt.figure(_fig_num)
                _title = (
                    (_fig._suptitle.get_text().strip() if _fig._suptitle else None)
                    or (_fig.axes[0].get_title().strip() if _fig.axes else None)
                    or f"fig_{_cnt[0]:02d}"
                )
                _safe = "".join(
                    c if c.isalnum() or c in "-_ " else "_" for c in _title
                ).strip().replace(" ", "_")[:60]
                _fname = os.path.join(OUT_DIR, f"{_safe}_{_cnt[0]:02d}.png")
                try:
                    _fig.savefig(_fname, dpi=110, bbox_inches="tight")
                    print(f"[capture] fig saved -> {_fname}")
                    _cnt[0] += 1
                except Exception as _save_err:
                    print(f"[capture] fig save failed: {_save_err}")
            return _orig_show(*a, **k)
        _show._wrapped = True
        _plt.show = _show
    print(f"[capture] matplotlib patched -> figures saved to {OUT_DIR}/")
except Exception as _cap_err:
    print("[capture] matplotlib not ready:", _cap_err)


**PS4 — Hourly Anomaly Detection**

---
SageMaker Studio | MLflow | Feature Store | Model Registry | Model Monitor

Architecture  : CUBIC MARS AWS v6
Target        : ensemble_anomaly_flag (binary: 1 if 2+ of 3 active signals)
Grain         : one row per DEVICE_ID per hour
Signals       : S1=SPC 3-sigma  S2=Isolation Forest  S3=Tap rejection >5%
                S4=SolarWinds network [INACTIVE — telemetry not available in Gold]
                S5=Bus ops ratio [REMOVED 2026-06-12 — CTA_REAL_TIME_BUS_DATA dropped]
NOTE          : signal_bus_ops (S5) and signal_network (S4) always default to 0.
                Effective ensemble = 2-of-3 active signals (S1, S2, S3).
Models        : 3-signal ensemble voting + HDBSCAN clustering + supervised XGBoost
Feasibility   : 95%  (5% gap: bus device IP coverage incomplete in SolarWinds)

---

In [ ]:
# ── CELL 1 : Install ──────────────────────────────────────────────────
import subprocess, sys
pkgs = [
    "s3fs","pyarrow",
    "scikit-learn>=1.4","xgboost>=2.0","lightgbm>=4.3",
    "umap-learn","optuna>=3.6","shap>=0.45","mlflow>=2.13", "sagemaker-mlflow",
    "boto3","sagemaker>=2.220,<3.0","pandas>=2.0","matplotlib","seaborn","scipy",
]
subprocess.check_call([sys.executable,"-m","pip","install","-q","--upgrade"]+pkgs)
print("OK")

In [ ]:
# ── CELL 2 : Imports ──────────────────────────────────────────────────
import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap, optuna, boto3, sagemaker, joblib, tarfile
from scipy import stats

# deltalake removed: gold tables exported as Parquet; pd.read_parquet() used in CELL 4
from sklearn.model_selection         import TimeSeriesSplit
from sklearn.preprocessing           import StandardScaler
from sklearn.ensemble                import (IsolationForest,
                                             RandomForestClassifier,
                                             StackingClassifier)
from sklearn.linear_model            import LogisticRegression
from sklearn.pipeline                import Pipeline
from sklearn.calibration             import CalibratedClassifierCV
from sklearn.metrics                 import (roc_auc_score, f1_score,
                                             average_precision_score,
                                             confusion_matrix,
                                             classification_report)
from xgboost   import XGBClassifier
from lightgbm  import LGBMClassifier, early_stopping, log_evaluation

import mlflow, mlflow.sklearn
from mlflow.models.signature import infer_signature
try:
    from sagemaker import Session
except ImportError:
    from sagemaker.session import Session                       # sagemaker >= 3
try:
    from sagemaker import get_execution_role
except ImportError:
    from sagemaker.session import get_execution_role            # sagemaker >= 3
from sagemaker.feature_store.feature_group      import FeatureGroup
from sagemaker.feature_store.feature_definition import (FeatureDefinition,
                                                        FeatureTypeEnum)
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
print("Imports OK")

In [ ]:
# ── CELL 3 : Configuration ──────────────────────────────────────────────────
ARTIFACT_BUCKET     = "cubic-mars-pm-s3-datalake-dev-artifacts-170202974600"
GOLD_BUCKET         = "cubic-mars-pm-s3-datalake-dev-gold-170202974600"
GOLD_PREFIX         = "chicago/gold/device_ps4_hourly"
GOLD_S3             = f"s3://{GOLD_BUCKET}/{GOLD_PREFIX}"

SM_SESSION          = Session()
ROLE                = get_execution_role()
REGION              = boto3.Session().region_name
SM_CLIENT           = boto3.client("sagemaker", region_name=REGION)
S3_CLIENT           = boto3.client("s3", region_name=REGION)

MLFLOW_SERVER_ARN = "arn:aws:sagemaker:us-east-1:170202974600:mlflow-tracking-server/cubic-mars-mlflow-server-dev"
os.environ["MLFLOW_TRACKING_SERVER_ARN"] = MLFLOW_SERVER_ARN

try:
    MLFLOW_TRACKING_URI = SM_CLIENT.describe_mlflow_tracking_server(
        TrackingServerName="cubic-mars-mlflow-server-dev"
    )["TrackingServerUrl"]
    print(f"MLflow managed server: {MLFLOW_TRACKING_URI}")
except Exception:
    MLFLOW_TRACKING_URI = "file:///home/sagemaker-user/mlruns"
    print("MLflow managed server not found — using local file fallback (EFS)")
mlflow.set_tracking_uri(MLFLOW_SERVER_ARN)  # sagemaker-mlflow resolves arn: scheme

EXPERIMENT_NAME     = "chicago-ps4-anomaly-detection"
MLFLOW_MODEL_NAME   = "ps4-anomaly-detection"
MODEL_REGISTRY_NAME = "chicago-ps4-anomaly-detection"
FS_GROUP_NAME       = "chicago-ps4-features"
ENDPOINT_NAME       = "chicago-ps4-anomaly-v1"
TARGET              = "ensemble_anomaly_flag"
SIGNAL_THRESHOLD    = 2   # 2-of-3 active signals (S4 network + S5 bus_ops inactive)

mlflow.set_experiment(EXPERIMENT_NAME)
print(f"Gold S3: {GOLD_S3}")

In [ ]:
# ── CELL 3b : Checkpoint — load saved state (skip Cells 4-8 if found) ────────
# Saves: df, train/val/test_df, X/y splits, FEATURE_COLS, SIGNAL_COLS, DEV_COL
# Path : /home/sagemaker-user/PS4/checkpoints/  (EFS — survives instance stop)
# Usage: Run Cells 1-3-3b → if CHECKPOINT_LOADED=True → jump to Cell 11 (SPC)
#        Set FORCE_RELOAD=True to re-pull from S3 and rebuild signals/features.

import os, joblib, numpy as _np_ckpt, pandas as _pd_ckpt

CKPT_DIR      = "/home/sagemaker-user/PS4/checkpoints"
FORCE_RELOAD  = False
CHECKPOINT_LOADED = False

_ckpt_df   = os.path.join(CKPT_DIR, "df.parquet")
_ckpt_meta = os.path.join(CKPT_DIR, "meta.joblib")

if not FORCE_RELOAD and os.path.exists(_ckpt_df) and os.path.exists(_ckpt_meta):
    print("[CHECKPOINT] Loading from", CKPT_DIR)
    df       = _pd_ckpt.read_parquet(_ckpt_df)
    train_df = _pd_ckpt.read_parquet(os.path.join(CKPT_DIR, "train_df.parquet"))
    val_df   = _pd_ckpt.read_parquet(os.path.join(CKPT_DIR, "val_df.parquet"))
    test_df  = _pd_ckpt.read_parquet(os.path.join(CKPT_DIR, "test_df.parquet"))
    _meta            = joblib.load(_ckpt_meta)
    FEATURE_COLS     = _meta["FEATURE_COLS"]
    SIGNAL_COLS      = _meta["SIGNAL_COLS"]
    TARGET           = _meta["TARGET"]
    DEV_COL          = _meta["DEV_COL"]
    DEVICE_TYPES     = _meta["DEVICE_TYPES"]
    SCALE_POS_WEIGHT = _meta["SCALE_POS_WEIGHT"]
    spc_results      = _meta.get("spc_results", {})
    _arrays          = joblib.load(os.path.join(CKPT_DIR, "arrays.joblib"))
    X_train = _arrays["X_train"]; y_train = _arrays["y_train"]
    X_val   = _arrays["X_val"];   y_val   = _arrays["y_val"]
    X_test  = _arrays["X_test"];  y_test  = _arrays["y_test"]
    X_cv    = _arrays["X_cv"];    y_cv    = _arrays["y_cv"]
    CHECKPOINT_LOADED = True
    print(f"  df           : {df.shape}")
    print(f"  train / val / test : {len(train_df):,} / {len(val_df):,} / {len(test_df):,}")
    print(f"  X_train      : {X_train.shape}  positives: {y_train.sum():,}")
    print(f"  FEATURE_COLS : {len(FEATURE_COLS)}  |  SIGNAL_COLS: {len(SIGNAL_COLS)}")
    print(f"  SCALE_POS_WEIGHT: {SCALE_POS_WEIGHT}")
    print()
    print("  CHECKPOINT LOADED — skip Cells 4-8.")
    print("  Jump to Cell 9 (Feature Store) or Cell 11 (SPC analysis).")
else:
    print("[CHECKPOINT] Not found or FORCE_RELOAD=True — running full pipeline (Cells 4-8).")
    print("  Checkpoint will be saved automatically at end of Cell 8.")


In [ ]:
# ── CELL 4 : Load Gold table (Parquet on S3) ──────────────────────────────────────────────────
print(f"Loading Gold table from: {GOLD_S3}")
df = pd.read_parquet(GOLD_S3, storage_options={"client_kwargs": {"region_name": REGION}})
GOLD_VERSION = None   # Parquet has no version; use None for MLflow tags
DATA_SOURCE  = "parquet"

# Gold table uses "hour_bucket" for the hourly timestamp; rename to hour_dt for downstream compat
if "hour_dt" not in df.columns and "hour_bucket" in df.columns:
    df = df.rename(columns={"hour_bucket": "hour_dt"})
df["hour_dt"] = pd.to_datetime(df["hour_dt"])

print(f"Parquet source  : confirmed")
print(f"Shape           : {df.shape}")
print(f"Date range      : {df['hour_dt'].min()}  →  {df['hour_dt'].max()}")
print(f"\nTarget distribution:\n{df[TARGET].value_counts()}")
print(f"Anomaly rate  : {df[TARGET].mean():.2%}")
print(f"\nData source: {DATA_SOURCE}  |  Gold version: {GOLD_VERSION}")

In [ ]:
# ── CELL 5 : Build signal ensemble flags ─────────────────────────────────────
# Gold pre-computes 3 signals from 28-day rolling baselines (event_rate_anomaly,
# metric_anomaly, reject_rate_anomaly).  Map these to PS4 signal names, then add
# TVM hardware signals (BHU/CHU/printer/OOS) so TVMs can reach ensemble threshold.
#
# Confirmed from Databricks (2026-06-30):
#   bhu/chu/printer/oos all present in Gold for TVM rows (1,784,896 rows)
#   bhu_nonzero_pct = 7.34%  ->  p90 must be computed only from TVM non-zero rows
#   oos_nonzero_pct = 34.68% ->  "any OOS event" is too broad; use p75 of non-zero

# ── Compute daily_event_count for SPC visualisation (Cell 12) ─────────────
if "event_count_hour" in df.columns and "transit_day" in df.columns:
    df["daily_event_count"] = df.groupby(
        ["DEVICE_ID", "transit_day"])["event_count_hour"].transform("sum")

# ── Signal 1: SPC event-rate anomaly (Gold 28-day baseline) ──────────────
if "event_rate_anomaly" in df.columns:
    df["signal_spc"] = df["event_rate_anomaly"].fillna(0).astype(int)
elif ("event_count_hour" in df.columns and "baseline_mean_events" in df.columns
      and "baseline_stddev_events" in df.columns):
    df["signal_spc"] = (
        (df["event_count_hour"] - df["baseline_mean_events"]).abs()
        > 3 * df["baseline_stddev_events"].fillna(1)
    ).astype(int)
else:
    df["signal_spc"] = 0

# ── Signal 2: Isolation Forest (placeholder filled in Cell 13) ──────────
if "signal_isolation_forest" not in df.columns:
    df["signal_isolation_forest"] = 0

# ── Signal 3: Tap rejection rate > 5% (GATE/VALIDATOR only — 0 for TVM) ─
if "tap_reject_rate_daily" in df.columns:
    df["signal_tap_rejection"] = (df["tap_reject_rate_daily"] > 5.0).astype(int)
elif "reject_rate_anomaly" in df.columns:
    df["signal_tap_rejection"] = df["reject_rate_anomaly"].fillna(0).astype(int)
else:
    df["signal_tap_rejection"] = 0

# ── Signal 4: Transaction-time latency anomaly (Gold metric_anomaly) ───────
if "metric_anomaly" in df.columns:
    df["signal_metric"] = df["metric_anomaly"].fillna(0).astype(int)
elif ("metric_401_avg_ms_hour" in df.columns
      and "baseline_mean_metric401" in df.columns
      and "baseline_stddev_metric401" in df.columns):
    df["signal_metric"] = (
        (df["metric_401_avg_ms_hour"] - df["baseline_mean_metric401"]).abs()
        > 2 * df["baseline_stddev_metric401"].fillna(1)
    ).astype(int)
else:
    df["signal_metric"] = 0

# ── Signal 5: Revenue-zero flag ───────────────────────────────────────────
if "use_revenue_zero_flag" in df.columns:
    df["signal_revenue_zero"] = df["use_revenue_zero_flag"].fillna(0).astype(int)
else:
    df["signal_revenue_zero"] = 0

# ── Signal 6: TVM hardware subsystem anomaly (BHU + CHU + PRINTER events) ─
# Fires when the sum of BHU+CHU+printer events in an hour is >= p90 of
# non-zero TVM hours only.  Reason: GATE/VALIDATOR always have hw_sum=0
# so a global p90 would be 0, and the condition would never fire.
# Confirmed: bhu_nonzero_pct=7.34% for TVM in Gold (2026-06-30).
_hw_cols = [c for c in ["bhu_count_hour", "chu_count_hour", "printer_count_hour"]
            if c in df.columns]
if _hw_cols:
    df["_hw_sum"] = df[_hw_cols].fillna(0).sum(axis=1)
    DEV_COL_TMP = next((c for c in df.columns
                        if c in ["mars_device_category","device_type","device_category"]), None)
    if DEV_COL_TMP:
        _tvm_mask  = df[DEV_COL_TMP].str.strip().str.upper() == "TVM"
        _tvm_nonzero = df.loc[_tvm_mask & (df["_hw_sum"] > 0), "_hw_sum"]
    else:
        _tvm_nonzero = df.loc[df["_hw_sum"] > 0, "_hw_sum"]
    if len(_tvm_nonzero) >= 50:
        _hw_p90 = _tvm_nonzero.quantile(0.90)
        df["signal_tvm_hw"] = (df["_hw_sum"] >= _hw_p90).astype(int)
        print(f"signal_tvm_hw: p90 of non-zero TVM HW events = {_hw_p90:.1f}")
    else:
        df["signal_tvm_hw"] = (df["_hw_sum"] > 0).astype(int)
        print("signal_tvm_hw: <50 TVM non-zero rows, using >0 threshold")
    df.drop(columns=["_hw_sum"], inplace=True)
else:
    df["signal_tvm_hw"] = 0
    print("signal_tvm_hw: bhu/chu/printer columns not found in Gold")

# ── Signal 7: Out-of-service event spike ──────────────────────────────────
# oos_count_hour > 0 fires for 34.68% of TVM hours (too broad: routine OOS logging).
# Use p75 of non-zero TVM oos hours to catch genuine OOS avalanches.
# Confirmed: oos_nonzero_pct=34.68% for TVM in Gold (2026-06-30).
if "oos_count_hour" in df.columns:
    _oos_nonzero = df.loc[df["oos_count_hour"] > 0, "oos_count_hour"]
    if len(_oos_nonzero) >= 50:
        _oos_p75 = _oos_nonzero.quantile(0.75)
        df["signal_oos"] = (df["oos_count_hour"] >= _oos_p75).astype(int)
        print(f"signal_oos:    p75 of non-zero OOS events = {_oos_p75:.1f}")
    else:
        df["signal_oos"] = (df["oos_count_hour"] > 0).astype(int)
else:
    df["signal_oos"] = 0
    print("signal_oos: oos_count_hour column not found")

# ── Ensemble: 2-of-N active signals ──────────────────────────────────────
SIGNAL_COLS = ["signal_spc", "signal_isolation_forest", "signal_tap_rejection",
               "signal_metric", "signal_revenue_zero", "signal_tvm_hw", "signal_oos"]
SIGNAL_COLS = [c for c in SIGNAL_COLS if c in df.columns]

df["signal_active_count"] = df[SIGNAL_COLS].sum(axis=1)
if TARGET not in df.columns or df[TARGET].sum() == 0:
    SIGNAL_THRESHOLD = 2
    df[TARGET] = (df["signal_active_count"] >= SIGNAL_THRESHOLD).astype(int)

# ── Device column detection ────────────────────────────────────────────────
DEV_COL = next((c for c in df.columns
                if c in ["mars_device_category", "device_type", "device_category"]), None)
DEVICE_TYPES = ["TVM", "GATE", "VALIDATOR"]

# ── Print summary ──────────────────────────────────────────────────────────
import pandas as _pd_sig
_SRC_MAP = {
    "signal_spc":             "event_rate_anomaly (Gold 28d baseline)",
    "signal_isolation_forest":"IF score — filled Cell 13",
    "signal_tap_rejection":   "tap_reject_rate_daily > 5%  [GATE/VALIDATOR]",
    "signal_metric":          "metric_anomaly — latency spike  [GATE/VALIDATOR]",
    "signal_revenue_zero":    "use_revenue_zero_flag  [GATE/VALIDATOR]",
    "signal_tvm_hw":          "BHU+CHU+printer >= p90 of non-zero TVM hrs  [TVM]",
    "signal_oos":             "oos_count_hour >= p75 of non-zero OOS hrs  [all]",
}
_rows = []
for s in SIGNAL_COLS:
    _row = {"Signal": s, "Source": _SRC_MAP.get(s, s),
            "Rate_All": f"{df[s].mean():.2%}",
            "Active_Rows": int(df[s].sum())}
    if DEV_COL:
        for _dt in DEVICE_TYPES:
            _m = df[df[DEV_COL].str.strip().str.upper()==_dt][s].mean()
            _row[f"Rate_{_dt}"] = f"{_m:.2%}"
    _rows.append(_row)
_rows.append({
    "Signal": TARGET, "Source": "ensemble (2+ signals)",
    "Rate_All": f"{df[TARGET].mean():.2%}",
    "Active_Rows": int(df[TARGET].sum()),
    **({f"Rate_{_dt}": f"{df[df[DEV_COL].str.strip().str.upper()==_dt][TARGET].mean():.2%}"
        for _dt in DEVICE_TYPES} if DEV_COL else {}),
})
print("\nSignal Summary:")
print(_pd_sig.DataFrame(_rows).to_string(index=False))

_neg = (df[TARGET]==0).sum(); _pos = max((df[TARGET]==1).sum(), 1)
print(f"\nscale_pos_weight hint: {round(_neg / _pos, 2)}")
if DEV_COL:
    print("")
    print(df.groupby(DEV_COL)[TARGET].agg(["count","mean"])
            .rename(columns={"count":"rows","mean":"anomaly_rate"}).to_string())


In [ ]:
# ── CELL 6 : Signal activation charts ──────────────────────────────────────────────────
import pandas as _pdv1

_sig_rates = {s: float(df[s].mean()) for s in SIGNAL_COLS if s in df.columns}
_sig_rates[TARGET] = float(df[TARGET].mean())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── 1. Horizontal bar: activation rate per signal ─────────────────────────
_sr = _pdv1.Series(_sig_rates).sort_values()
_colors = ["#E53935" if k == TARGET else "#1565C0" for k in _sr.index]
_sr.plot(kind="barh", ax=axes[0], color=_colors, edgecolor="white")
axes[0].axvline(0.05, color="orange", ls="--", lw=1.2, label="5% ref")
for bar, v in zip(axes[0].patches, _sr.values):
    axes[0].text(v + 0.001, bar.get_y() + bar.get_height()/2,
                 f"{v:.2%}", va="center", fontsize=8)
axes[0].set_title("Signal activation rates (all rows)", fontsize=11, fontweight="bold")
axes[0].set_xlabel("Activation rate"); axes[0].legend()

# ── 2. Device Ã— signal heatmap ─────────────────────────────────────────────
if DEV_COL and DEV_COL in df.columns:
    _hmap = df.groupby(DEV_COL)[SIGNAL_COLS + [TARGET]].mean()
    sns.heatmap(_hmap.T, ax=axes[1], cmap="YlOrRd", annot=True, fmt=".1%",
                linewidths=0.5, cbar_kws={"label": "rate"})
    axes[1].set_title("Signal rate by device type", fontsize=11, fontweight="bold")
    axes[1].set_xlabel("Device type"); axes[1].set_ylabel("")
else:
    axes[1].text(0.5, 0.5, "No device column", ha="center", va="center",
                 transform=axes[1].transAxes)
    axes[1].set_title("Device Ã— signal heatmap")

# ── 3. Signal co-activation matrix ─────────────────────────────────────────
_active = df[SIGNAL_COLS].astype(float)
_co = _active.T.dot(_active)  # count of rows where both signals are active
_co_pct = _co.div(_co.values.diagonal(), axis=0)  # normalise by row signal count
np.fill_diagonal(_co_pct.values, np.nan)
sns.heatmap(_co_pct, ax=axes[2], cmap="Blues", annot=True, fmt=".0%",
            linewidths=0.5, mask=_co_pct.isna(),
            cbar_kws={"label":"% of row signal rows where col also fires"})
axes[2].set_title("Signal co-activation matrix", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.savefig("/tmp/ps4_signal_charts.png", dpi=100); plt.show()
with mlflow.start_run(run_name="signal_activation_viz") as _r:
    mlflow.set_tags({**globals().get("COMMON_TAGS", {"ps":"PS4","target":"ensemble_anomaly_flag","city":"chicago"}), "model_type":"visualization"})
    mlflow.log_artifact("/tmp/ps4_signal_charts.png")

# ── Summary table ─────────────────────────────────────────────────────────
print("\n── Signal Summary Table ──")
_rows = []
for s in SIGNAL_COLS + [TARGET]:
    if s not in df.columns: continue
    _col = df[s]
    _row = {"Signal": s, "Rate_All": f"{_col.mean():.2%}",
            "Active_Rows": int(_col.sum())}
    if DEV_COL and DEV_COL in df.columns:
        for _dt in DEVICE_TYPES:
            _m = df[df[DEV_COL].str.strip().str.upper()==_dt][s].mean()
            _row[f"Rate_{_dt}"] = f"{_m:.2%}"
    _rows.append(_row)
print(_pdv1.DataFrame(_rows).to_string(index=False))


In [ ]:
# ── CELL 7 : EDA ──────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 9))

# Signal co-activation heatmap
if len(SIGNAL_COLS) >= 2:
    sns.heatmap(df[SIGNAL_COLS].corr(), annot=True, fmt=".2f", cmap="RdBu_r",
                center=0, ax=axes[0,0], linewidths=0.5)
    axes[0,0].set_title("Signal Co-activation Correlation")

# Anomaly flag distribution
df[TARGET].value_counts().plot(kind="bar", ax=axes[0,1],
    color=["#43A047","#E53935"], edgecolor="white")
axes[0,1].set_title("ensemble_anomaly_flag distribution")

# ── Temporal: hour-of-day Ã— day-of-week heatmap ───────────────────────────
_hod = df["hour_dt"].dt.hour
_dow = df["hour_dt"].dt.day_name()
_DOW_ORDER = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
_pivot = (df.assign(hour=_hod, dow=_dow)
            .groupby(["dow","hour"])[TARGET].mean()
            .unstack(fill_value=0))
_pivot = _pivot.reindex([d for d in _DOW_ORDER if d in _pivot.index])
sns.heatmap(_pivot, ax=axes[0,2], cmap="YlOrRd", fmt=".0%",
            annot=True, linewidths=0.3, cbar_kws={"label":"anomaly rate"})
axes[0,2].set_title("Anomaly rate: hour Ã— day-of-week")
axes[0,2].set_xlabel("Hour of day"); axes[0,2].set_ylabel("")

# ── Monthly anomaly rate trend ─────────────────────────────────────────────
_monthly = (df.assign(month=df["hour_dt"].dt.to_period("M"))
              .groupby("month")[TARGET].mean())
_monthly.index = _monthly.index.astype(str)
axes[1,0].plot(_monthly.index, _monthly.values, marker="o", color="#1565C0", lw=2)
axes[1,0].set_title("Monthly anomaly rate trend")
axes[1,0].set_xlabel("Month"); axes[1,0].set_ylabel("Anomaly rate")
axes[1,0].tick_params(axis="x", rotation=45)
axes[1,0].axhline(_monthly.mean(), color="red", ls="--", lw=1, label="mean")
axes[1,0].legend()

# Hourly pattern
df.groupby(_hod)[TARGET].mean().plot(kind="bar", ax=axes[1,1], color="#1565C0")
axes[1,1].set_title("Anomaly rate by hour of day")

# By device type
if DEV_COL and DEV_COL in df.columns:
    df.groupby(DEV_COL)[TARGET].mean().sort_values(ascending=False).plot(
        kind="bar", ax=axes[1,2], color="#7B1FA2")
    axes[1,2].set_title("Anomaly rate by device type")
else:
    axes[1,2].text(0.5, 0.5, "No device column", ha="center", va="center",
                   transform=axes[1,2].transAxes)

plt.tight_layout()
plt.savefig("/tmp/ps4_eda.png", dpi=100); plt.show()


In [ ]:
# ── CELL 8 : Feature engineering & splits ──────────────────────────────────────────────────
# Raw gold signal components (event_rate_anomaly/metric_anomaly/reject_rate_anomaly) are
# pre-computed 2-sigma flags that directly compose ensemble_anomaly_flag (the target).
# Including them as features would be near-complete leakage — the model would trivially
# learn "2+ signals → flag=1" instead of finding leading indicators. Exclude from FEATURE_COLS.
RAW_SIGNAL_COLS = ["event_rate_anomaly", "metric_anomaly", "reject_rate_anomaly"]
ID_COLS = ["DEVICE_ID","hour_dt","split"] + SIGNAL_COLS + ["signal_active_count"] + RAW_SIGNAL_COLS
ID_COLS = [c for c in ID_COLS if c in df.columns]

FEATURE_COLS = [
    c for c in df.columns
    if c not in ID_COLS + [TARGET]
    and pd.api.types.is_numeric_dtype(df[c])
]
df[FEATURE_COLS] = df[FEATURE_COLS].fillna(df[FEATURE_COLS].median())
print(f"Features: {len(FEATURE_COLS)}")

train_df = df[df["hour_dt"] <  "2025-10-01"]
val_df   = df[(df["hour_dt"] >= "2025-10-01") & (df["hour_dt"] < "2026-01-01")]
test_df  = df[df["hour_dt"] >= "2026-01-01"]

X_train, y_train = train_df[FEATURE_COLS].values, train_df[TARGET].values
X_val,   y_val   = val_df[FEATURE_COLS].values,   val_df[TARGET].values
X_test,  y_test  = test_df[FEATURE_COLS].values,  test_df[TARGET].values
X_cv             = np.vstack([X_train,X_val])
y_cv             = np.concatenate([y_train,y_val])

neg, pos         = (y_train==0).sum(), (y_train==1).sum()
SCALE_POS_WEIGHT = round(neg/pos, 2)
print(f"Train:{len(X_train):,}  Val:{len(X_val):,}  Test:{len(X_test):,}")
print(f"scale_pos_weight: {SCALE_POS_WEIGHT}")

# ── Save checkpoint after feature engineering & splits ───────────────────
import os, joblib as _jl
_ckpt_dir = "/home/sagemaker-user/PS4/checkpoints"
os.makedirs(_ckpt_dir, exist_ok=True)
df.to_parquet(       os.path.join(_ckpt_dir, "df.parquet"),       index=True)
train_df.to_parquet( os.path.join(_ckpt_dir, "train_df.parquet"), index=True)
val_df.to_parquet(   os.path.join(_ckpt_dir, "val_df.parquet"),   index=True)
test_df.to_parquet(  os.path.join(_ckpt_dir, "test_df.parquet"),  index=True)
_jl.dump({
    "FEATURE_COLS":     FEATURE_COLS,
    "SIGNAL_COLS":      SIGNAL_COLS,
    "TARGET":           TARGET,
    "DEV_COL":          DEV_COL,
    "DEVICE_TYPES":     DEVICE_TYPES,
    "SCALE_POS_WEIGHT": SCALE_POS_WEIGHT,
    "spc_results":      spc_results if "spc_results" in dir() else {},
}, os.path.join(_ckpt_dir, "meta.joblib"))
_jl.dump({
    "X_train": X_train, "y_train": y_train,
    "X_val":   X_val,   "y_val":   y_val,
    "X_test":  X_test,  "y_test":  y_test,
    "X_cv":    X_cv,    "y_cv":    y_cv,
}, os.path.join(_ckpt_dir, "arrays.joblib"))
print(f"[CHECKPOINT] Saved to {_ckpt_dir}")
print(f"  df={df.shape}  X_train={X_train.shape}  positives={y_train.sum():,}")
print(f"  Next kernel restart: run Cells 1-3-3b only, then jump to Cell 11.")

In [ ]:
# ── CELL 9 : Feature Store ──────────────────────────────────────────────────
# NOTE: This is the "Cell 9" seen in Jupyter UI (markdown cell at [0] shifts count).
# Set ENABLE_FEATURE_STORE=False to skip entirely during dev reruns — saves ~5 min.
ENABLE_FEATURE_STORE = False

if not ENABLE_FEATURE_STORE:
    print("[SKIP] Feature Store disabled — set ENABLE_FEATURE_STORE=True to enable")
else:
    # Ingest last 90 days of training data — representative for online serving,
    # avoids ingesting millions of rows from years of history.
    _cutoff_fs = train_df["hour_dt"].max() - pd.Timedelta(days=90)
    fs_df = train_df[train_df["hour_dt"] >= _cutoff_fs][
        FEATURE_COLS + [TARGET, "DEVICE_ID", "hour_dt"]].copy()
    fs_df["event_time"] = fs_df["hour_dt"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    fs_df["DEVICE_ID"]  = fs_df["DEVICE_ID"].astype(str)
    fs_df               = fs_df.drop(columns=["hour_dt"])
    print(f"Feature Store sample: {len(fs_df):,} rows (last 90 days of train)")

    def dtype_to_fs(dtype):
        if pd.api.types.is_float_dtype(dtype):   return FeatureTypeEnum.FRACTIONAL
        if pd.api.types.is_integer_dtype(dtype): return FeatureTypeEnum.INTEGRAL
        return FeatureTypeEnum.STRING

    feature_group = FeatureGroup(name=FS_GROUP_NAME, sagemaker_session=SM_SESSION,
        feature_definitions=[
            FeatureDefinition(feature_name=c, feature_type=dtype_to_fs(fs_df[c].dtype))
            for c in fs_df.columns])

    _sm_fg_client = SM_SESSION.boto_session.client("sagemaker", region_name=REGION)
    _deleted = False
    try:
        _sm_fg_client.delete_feature_group(FeatureGroupName=FS_GROUP_NAME)
        print(f"Deleted existing Feature Group: {FS_GROUP_NAME}")
        _deleted = True
    except Exception:
        pass  # did not exist — fine

    if _deleted:
        print("Waiting 60s for deletion to propagate ...")
        time.sleep(60)   # only sleep when we actually deleted something

    try:
        feature_group.create(
            s3_uri=f"s3://{ARTIFACT_BUCKET}/feature-store/{FS_GROUP_NAME}",
            record_identifier_name="DEVICE_ID", event_time_feature_name="event_time",
            role_arn=ROLE, enable_online_store=True,
            description="PS4 hourly anomaly features — Chicago Ventra")
        _status = ""
        while _status != "Created":
            time.sleep(10)  # 10s instead of 15s
            _status = feature_group.describe().get("FeatureGroupStatus")
            print(f"  Feature Group status: {_status}")
    except Exception as _fg_e:
        print(f"Feature Group create: {_fg_e}")

    feature_group.ingest(
        data_frame=fs_df.reset_index(drop=True),
        max_workers=8,   # doubled from 4
        wait=True)
    print(f"Ingested {len(fs_df):,} → {FS_GROUP_NAME}")


In [ ]:
# ── CELL 10 : Helpers ──────────────────────────────────────────────────
COMMON_TAGS={"ps":"PS4","target":"ensemble_anomaly_flag","city":"chicago",
             "gold_version":str(GOLD_VERSION),"architecture":"CUBIC-MARS-AWS-v6"}

def evaluate(model,X,y,split="test"):
    prob=model.predict_proba(X)[:,1]; pred=(prob>=0.5).astype(int)
    return {f"{split}_auc" :round(roc_auc_score(y,prob),4),
            f"{split}_ap"  :round(average_precision_score(y,prob),4),
            f"{split}_f1"  :round(f1_score(y,pred),4)}

def log_cm(model,X,y,name):
    pred=(model.predict_proba(X)[:,1]>=0.5).astype(int)
    cm=confusion_matrix(y,pred)
    fig,ax=plt.subplots(figsize=(4,3))
    sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",ax=ax,
                xticklabels=["Normal","Anomaly"],
                yticklabels=["Normal","Anomaly"])
    ax.set_title(name); fig.tight_layout()
    fig.savefig(f"/tmp/cm_{name}.png",dpi=80); plt.close(fig)
    mlflow.log_artifact(f"/tmp/cm_{name}.png")

In [ ]:
# -- CELL 11 : Signal 1 -- SPC analysis + device/facility breakdown + Option A fix
# signal_spc = event_rate_anomaly: per-device 2-sigma hourly flag (Gold 28-day baseline).
# violation_device_hours: count of (device x hour) rows where signal_spc=1 on that day.
#   NOT "out of service" -- device may be fully operational with unusual event volume.
# avg_event_count: mean transactions/events per device per day (busyness, not downtime).
# fleet_spc_rate: fraction of fleet device-hours firing; >5% = real fleet-wide spike.

import pandas as _pdsp, json as _json11
import matplotlib.pyplot as _plt11
import numpy as _np11

spc_results  = {}
anomaly_days = {}   # {device_type: DataFrame -- date / violation_device_hours / avg_event_count}

if "daily_event_count" in df.columns:
    _dev_groups = ([(dt, df[df[DEV_COL].str.strip().str.upper()==dt])
                    for dt in DEVICE_TYPES]
                   if DEV_COL else [("ALL", df)])
    for _dt_name, _subset in _dev_groups:
        _s = _subset["daily_event_count"].dropna()
        if len(_s) < 50: continue
        _mean, _std = _s.mean(), _s.std()
        spc_results[_dt_name] = {
            "mean": round(_mean, 4),
            "ucl":  round(_mean + 3 * _std, 4),
            "lcl":  max(0, round(_mean - 3 * _std, 4)),
        }
        if "signal_spc" in _subset.columns and "hour_dt" in _subset.columns:
            _sub2 = _subset.copy()
            _sub2["_date"] = _sub2["hour_dt"].dt.date
            _dv = _sub2.groupby("_date")["signal_spc"].sum()
            _de = (_sub2.groupby("_date")["daily_event_count"].mean()
                   if "daily_event_count" in _sub2.columns else None)
            _anom = _dv[_dv > 0].reset_index()
            _anom.columns = ["date", "violation_device_hours"]
            if _de is not None:
                _anom["avg_event_count"] = _anom["date"].map(_de).round(1)
            _anom["device_type"] = _dt_name
            _anom["date"] = _pdsp.to_datetime(_anom["date"])
            _anom["month"] = _anom["date"].dt.to_period("M").astype(str)
            anomaly_days[_dt_name] = _anom.sort_values(
                "violation_device_hours", ascending=False).reset_index(drop=True)
elif "event_rate_anomaly" in df.columns and DEV_COL:
    for _dt_name in DEVICE_TYPES:
        _s = df[df[DEV_COL].str.strip().str.upper()==_dt_name]
        if len(_s) < 50: continue
        _mean = _s["baseline_mean_events"].mean() if "baseline_mean_events" in df.columns else 0
        _std  = _s["baseline_stddev_events"].mean() if "baseline_stddev_events" in df.columns else 1
        spc_results[_dt_name] = {
            "mean": round(_mean, 4),
            "ucl":  round(_mean + 3 * _std, 4),
            "lcl":  max(0, round(_mean - 3 * _std, 4)),
        }
else:
    print("[INFO] daily_event_count not found -- SPC thresholds skipped.")

# -- MLflow: log thresholds
with mlflow.start_run(run_name="spc_signal1_analysis") as run:
    mlflow.set_tags({**COMMON_TAGS, "model_type": "spc_control_chart", "signal": "S1"})
    for dt_name, vals in spc_results.items():
        mlflow.log_metric(f"{dt_name}_mean", vals["mean"])
        mlflow.log_metric(f"{dt_name}_ucl",  vals["ucl"])
        mlflow.log_metric(f"{dt_name}_lcl",  vals["lcl"])
        if dt_name in anomaly_days:
            mlflow.log_metric(f"{dt_name}_anomalous_days", len(anomaly_days[dt_name]))
    with open("/tmp/spc_thresholds.json", "w") as _f:
        _json11.dump(spc_results, _f, indent=2)
    mlflow.log_artifact("/tmp/spc_thresholds.json")
    mlflow.log_param("sigma_threshold", 3)
    SPC_RUN = run.info.run_id

# -- Day-level summary
_SEP = "=" * 65
print(f"\n{_SEP}")
print("SPC Signal 1 -- Fleet Anomaly Summary")
print("  violation_device_hours = (device x hour) rows with signal_spc=1 -- NOT out-of-service")
print("  avg_event_count        = mean transactions per device per day (busyness indicator)")
print("  99%+ days flagged: per-device 2-sigma fires ~5%/device-hr -> Option A below fixes this")
print(_SEP)
for _dt_name, _vals in spc_results.items():
    _mask = (df[DEV_COL].str.strip().str.upper() == _dt_name
             if DEV_COL in df.columns else _pdsp.Series([True]*len(df)))
    _total_days  = df.loc[_mask, "hour_dt"].dt.date.nunique() if "hour_dt" in df.columns else 0
    _adf         = anomaly_days.get(_dt_name, _pdsp.DataFrame())
    _anom_cnt    = len(_adf)
    _viol_rate   = _anom_cnt / _total_days * 100 if _total_days > 0 else 0
    _total_viol  = int(_adf["violation_device_hours"].sum()) if len(_adf) else 0
    _lcl  = _vals["lcl"];  _mn = _vals["mean"];  _ucl = _vals["ucl"]
    print(f"\n  {_dt_name}")
    print(f"    Control limits  : LCL={_lcl:.1f}  Mean={_mn:.1f}  UCL={_ucl:.1f}")
    print(f"    Total days      : {_total_days:,}")
    print(f"    Days with any device firing     : {_anom_cnt:,} ({_viol_rate:.1f}%)")
    print(f"    Total violation device-hours    : {_total_viol:,}")
    if _anom_cnt > 0:
        print(f"    Top 5 days by violation device-hours:")
        for _, _row in _adf.head(5).iterrows():
            _ec = f"  avg_events/device={_row['avg_event_count']:.0f}" if "avg_event_count" in _row else ""
            print(f"      {str(_row['date'])[:10]}  --  {int(_row['violation_device_hours']):,} device-hrs{_ec}")

# -- Device-level top 5 per device type
_dev_id_col = next((c for c in ["DEVICE_ID", "device_id"] if c in df.columns), None)
_dev_nm_col = next((c for c in ["DEVICE_NAME", "device_name"] if c in df.columns), None)
if "signal_spc" in df.columns and _dev_id_col and "hour_dt" in df.columns:
    print(f"\n{_SEP}")
    print("Top 5 Devices by Violation Device-Hours (per device type)")
    print(_SEP)
    for _dt_name in DEVICE_TYPES:
        if _dt_name not in spc_results: continue
        _mask = df[DEV_COL].str.strip().str.upper() == _dt_name
        _grp  = df[_mask].groupby(_dev_id_col)["signal_spc"].sum().sort_values(ascending=False)
        _top5 = _grp.head(5).reset_index()
        _top5.columns = ["DEVICE_ID", "violation_device_hours"]
        if _dev_nm_col:
            _nm_map = (df[[_dev_id_col, _dev_nm_col]].drop_duplicates()
                          .set_index(_dev_id_col)[_dev_nm_col])
            _top5["DEVICE_NAME"] = _top5["DEVICE_ID"].map(_nm_map)
        _fleet_total = int(df[_mask]["signal_spc"].sum())
        _fleet_devs  = int(df[_mask][_dev_id_col].nunique())
        _pct_top5 = (_top5["violation_device_hours"].sum() / _fleet_total * 100
                     if _fleet_total > 0 else 0)
        print(f"\n  {_dt_name}  ({_fleet_devs} devices | {_fleet_total:,} total violation device-hrs)")
        print(_top5.to_string(index=False))
        print(f"    Top 5 account for {_pct_top5:.1f}% of all {_dt_name} violations")

# -- Facility-level top 5 per device type
_fac_id_col = next((c for c in ["FACILITY_ID","facility_id","facid"] if c in df.columns), None)
_fac_nm_col = next((c for c in ["FACILITY_NAME","facility_name"] if c in df.columns), None)
if "signal_spc" in df.columns and _fac_id_col and DEV_COL in df.columns:
    print(f"\n{_SEP}")
    print("Top 5 Facilities by Violation Device-Hours (per device type)")
    print(_SEP)
    for _dt_name in DEVICE_TYPES:
        if _dt_name not in spc_results: continue
        _mask  = df[DEV_COL].str.strip().str.upper() == _dt_name
        _gcols = [_fac_id_col] + ([_fac_nm_col] if _fac_nm_col else [])
        _grp   = (df[_mask].groupby(_gcols)["signal_spc"]
                            .sum().sort_values(ascending=False).head(5).reset_index())
        _grp.columns = _gcols + ["violation_device_hours"]
        _fac_total = int(df[_mask]["signal_spc"].sum())
        print(f"\n  {_dt_name}  ({_fac_total:,} total violation device-hrs)")
        print(_grp.to_string(index=False))

# -- Monthly heatmap
if anomaly_days:
    _all = _pdsp.concat(anomaly_days.values(), ignore_index=True)
    if "month" in _all.columns and len(_all) > 0:
        _pivot = (_all.groupby(["device_type","month"])["violation_device_hours"]
                      .sum().unstack(fill_value=0))
        _pivot = _pivot.reindex(columns=sorted(_pivot.columns))
        fig11, ax11 = _plt11.subplots(figsize=(max(12, len(_pivot.columns)*0.9),
                                                max(3, len(_pivot)*1.2)))
        _im = ax11.imshow(_pivot.values, aspect="auto", cmap="YlOrRd")
        ax11.set_xticks(range(len(_pivot.columns)))
        ax11.set_xticklabels(_pivot.columns, rotation=45, ha="right", fontsize=8)
        ax11.set_yticks(range(len(_pivot.index)))
        ax11.set_yticklabels(_pivot.index, fontsize=9)
        for _i in range(len(_pivot.index)):
            for _j in range(len(_pivot.columns)):
                _v = int(_pivot.values[_i, _j])
                ax11.text(_j, _i, f"{_v:,}" if _v > 0 else "",
                          ha="center", va="center", fontsize=6,
                          color="white" if _v > _pivot.values.max()*0.6 else "black")
        _plt11.colorbar(_im, ax=ax11, label="Violation device-hours / month")
        ax11.set_title("SPC Violation Device-Hours -- Monthly Heatmap by Device Type",
                       fontsize=11, fontweight="bold")
        _plt11.tight_layout()
        _plt11.savefig("/tmp/ps4_spc_heatmap.png", dpi=100)
        _plt11.show()
        with mlflow.start_run(run_name="spc_monthly_heatmap") as _r:
            mlflow.set_tags({**COMMON_TAGS, "model_type": "visualization"})
            mlflow.log_artifact("/tmp/ps4_spc_heatmap.png")

# -- Option A: Fleet-level SPC signal (>5% of device-hours = real fleet spike)
# Replaces per-device 2-sigma signal in ensemble. Per-device signal_spc kept for
# device/facility tables above; fleet signal is what ML models should learn from.
if "signal_spc" in df.columns and DEV_COL in df.columns and "hour_dt" in df.columns:
    print(f"\n{_SEP}")
    print("Option A -- Fleet-level SPC Signal")
    print("  Per-device 2-sigma fires ~5% of device-hours by design (Gold 28-day baseline).")
    print("  Large fleet -> nearly every day has >=1 device firing -> 99% days anomalous.")
    print("  Fix: flag (device_type, day) only if >5% of fleet device-hours fire signal_spc.")
    print(_SEP)

    _orig_spc_rate = float(df["signal_spc"].mean())
    print(f"  Original signal_spc rate : {_orig_spc_rate:.1%} of all device-hours")

    # Fleet rate per (device_type, day) via groupby transform -- same index as df
    df["_date_tmp"] = df["hour_dt"].dt.date.astype(str)
    df["fleet_spc_rate"] = df.groupby([DEV_COL, "_date_tmp"])["signal_spc"].transform("mean")
    _FLEET_THRESHOLD = 0.05   # 5% of fleet = 2x expected 2-sigma baseline rate
    df["signal_spc_fleet"] = (df["fleet_spc_rate"] > _FLEET_THRESHOLD).astype(int)
    df.drop(columns=["_date_tmp"], inplace=True, errors="ignore")

    _new_spc_rate = float(df["signal_spc_fleet"].mean())
    print(f"  Fleet signal rate (>5%)  : {_new_spc_rate:.1%} of all device-hours")
    print()
    print("  Per device type:")
    for _dt in DEVICE_TYPES:
        _mask = df[DEV_COL].str.strip().str.upper() == _dt
        _o = df.loc[_mask, "signal_spc"].mean()
        _n = df.loc[_mask, "signal_spc_fleet"].mean()
        # Count anomalous days (fleet signal = 1 for any hour that day)
        _anom_d = (df.loc[_mask].groupby(df.loc[_mask,"hour_dt"].dt.date)["signal_spc_fleet"]
                   .max().sum())
        _total_d = df.loc[_mask, "hour_dt"].dt.date.nunique()
        print(f"    {_dt}: device-hr rate {_o:.1%} -> {_n:.1%}"
              f"  |  anomalous days {_anom_d}/{_total_d} ({_anom_d/_total_d*100:.1f}%)"
              if _total_d > 0 else f"    {_dt}: device-hr rate {_o:.1%} -> {_n:.1%}")

    # Apply Option A: replace signal_spc in ensemble with fleet version
    df["signal_spc"] = df["signal_spc_fleet"]
    df["signal_active_count"] = df[SIGNAL_COLS].sum(axis=1)
    df[TARGET] = (df["signal_active_count"] >= SIGNAL_THRESHOLD).astype(int)
    _new_ens = float(df[TARGET].mean())
    print(f"\n  Ensemble anomaly rate after Option A: {_new_ens:.2%}"
          f"  (was {_orig_spc_rate:.2%} before fleet fix)")
    print("  fleet_spc_rate added as continuous feature for ML models.")
    if "fleet_spc_rate" not in FEATURE_COLS:
        FEATURE_COLS.append("fleet_spc_rate")
    # Propagate fleet_spc_rate to train/val/test splits (splits were created before
    # Option A ran, so they do not have this column yet).
    for _sdf in [train_df, val_df, test_df]:
        _sdf["fleet_spc_rate"] = df.loc[_sdf.index, "fleet_spc_rate"]

    with mlflow.start_run(run_name="spc_option_a_fleet_signal") as _r:
        mlflow.set_tags({**COMMON_TAGS, "model_type": "signal_engineering"})
        mlflow.log_param("fleet_threshold_pct", _FLEET_THRESHOLD * 100)
        mlflow.log_metric("original_spc_device_hr_rate", round(_orig_spc_rate, 4))
        mlflow.log_metric("fleet_spc_device_hr_rate",    round(_new_spc_rate, 4))
        mlflow.log_metric("ensemble_rate_after_optionA", round(_new_ens, 4))

    # Persist updated df + y arrays to EFS checkpoint
    import joblib as _jla
    df.to_parquet(os.path.join(CKPT_DIR, "df.parquet"), index=True)
    _arr_path = os.path.join(CKPT_DIR, "arrays.joblib")
    if os.path.exists(_arr_path):
        _arrs = _jla.load(_arr_path)
        _arrs["y_train"] = train_df.join(df[[TARGET]], rsuffix="_r")[TARGET].values
        _arrs["y_val"]   = val_df.join(df[[TARGET]],   rsuffix="_r")[TARGET].values
        _arrs["y_test"]  = test_df.join(df[[TARGET]],  rsuffix="_r")[TARGET].values
        _arrs["y_cv"]    = _np11.concatenate([_arrs["y_train"], _arrs["y_val"]])
        _jla.dump(_arrs, _arr_path)
    print("[CHECKPOINT] Updated df.parquet + arrays.joblib with Option A fleet signal.")

# -- Save daily SPC data + anomaly_days for Cell 12
import joblib as _jl11, os as _os11
_SPC_DAILY_CKPT = os.path.join(CKPT_DIR, "spc_daily.joblib")
_spc_cache11 = {}
if spc_results and "hour_dt" in df.columns and DEV_COL in df.columns:
    print("\n[CHECKPOINT] Pre-aggregating daily SPC data for Cell 12 ...")
    for _dt11 in [d for d in DEVICE_TYPES if d in spc_results]:
        _mask11   = df[DEV_COL].str.strip().str.upper() == _dt11
        _date_idx = _pdsp.to_datetime(df.loc[_mask11, "hour_dt"].dt.date)
        _spc_cache11[_dt11] = {
            "daily_mean": (df.loc[_mask11, "daily_event_count"].groupby(_date_idx).mean()
                           if "daily_event_count" in df.columns else None),
            "viol_sum":   (df.loc[_mask11, "signal_spc"].groupby(_date_idx).sum()
                           if "signal_spc" in df.columns else None),
        }
    _ts11_cols = [s for s in SIGNAL_COLS + [TARGET] if s in df.columns]
    _sig_ts11  = None
    if _ts11_cols and "hour_dt" in df.columns:
        _sig_ts11 = (df[_ts11_cols]
                     .groupby(_pdsp.to_datetime(df["hour_dt"].dt.date)).mean()
                     .rolling(7, min_periods=1).mean())
    _jl11.dump({"spc_daily": _spc_cache11, "signal_ts": _sig_ts11,
                "ts_cols": _ts11_cols, "anomaly_days": anomaly_days}, _SPC_DAILY_CKPT)
    print(f"[CHECKPOINT] spc_daily + anomaly_days saved -> {_SPC_DAILY_CKPT}")

_meta11_path = os.path.join(CKPT_DIR, "meta.joblib")
if _os11.path.exists(_meta11_path):
    _m11 = _jl11.load(_meta11_path)
    _m11["spc_results"]  = spc_results
    _m11["anomaly_days"] = anomaly_days
    _jl11.dump(_m11, _meta11_path)
    print("[CHECKPOINT] spc_results + anomaly_days persisted to meta.joblib")


In [ ]:
# -- CELL 12 : SPC control charts + anomalous days + device/facility breakdown + signal timeline
# Loads pre-aggregated daily data from Cell 11 checkpoint (fast).
# Shows: control chart per device type, anomalous day markers, days table,
#        top devices/facilities driving violations, signal timeline.
import pandas as _pdv2, joblib as _jl12, os as _os12

_SPC_DAILY_CKPT  = os.path.join(CKPT_DIR, "spc_daily.joblib")
_spc_day_cache   = {}
_signal_ts_cache = None
_signal_ts_cols  = []
_anom_days_cache = {}

if _os12.path.exists(_SPC_DAILY_CKPT):
    _ck12 = _jl12.load(_SPC_DAILY_CKPT)
    _spc_day_cache   = _ck12.get("spc_daily", {})
    _signal_ts_cache = _ck12.get("signal_ts")
    _signal_ts_cols  = _ck12.get("ts_cols", [])
    _anom_days_cache = _ck12.get("anomaly_days", {})
    print("[CHECKPOINT] Daily SPC data loaded -- skipping 30M-row resample.")
else:
    print("[INFO] No spc_daily checkpoint -- run Cell 11 first.")

# -- Per-device SPC control chart with anomalous day markers
if "daily_event_count" in df.columns and DEV_COL and spc_results:
    _devs = [d for d in DEVICE_TYPES if d in spc_results]
    if _devs and "hour_dt" in df.columns:
        fig, axes = plt.subplots(len(_devs), 1, figsize=(18, 5*len(_devs)))
        if len(_devs) == 1: axes = [axes]
        for ax, _dt in zip(axes, _devs):
            if _dt in _spc_day_cache and _spc_day_cache[_dt]["daily_mean"] is not None:
                _daily      = _spc_day_cache[_dt]["daily_mean"].dropna()
                _viol_daily = _spc_day_cache[_dt]["viol_sum"].dropna()
            else:
                _sub = df[df[DEV_COL].str.strip().str.upper() == _dt].set_index("hour_dt").sort_index()
                _daily      = _sub["daily_event_count"].resample("1D").mean().dropna()
                _viol_daily = _sub["signal_spc"].resample("1D").sum().dropna()

            _ucl = spc_results[_dt]["ucl"]
            _lcl = spc_results[_dt]["lcl"]
            _mn  = spc_results[_dt]["mean"]

            ax.plot(_daily.index, _daily.values,
                    lw=1.0, alpha=0.85, color="#1565C0", label="Daily mean event count")
            ax.axhline(_ucl, color="#D32F2F", ls="--", lw=1.5, label=f"UCL = {_ucl:.0f}")
            ax.axhline(_lcl, color="#1976D2", ls="--", lw=1.5, label=f"LCL = {_lcl:.0f}")
            ax.axhline(_mn,  color="#2E7D32", ls="-",  lw=1.5, label=f"Mean = {_mn:.0f}")
            ax.fill_between(_daily.index, _ucl, _daily.values,
                            where=(_daily.values > _ucl),
                            alpha=0.15, color="#D32F2F", label="Above UCL")
            ax.fill_between(_daily.index, _lcl, _daily.values,
                            where=(_daily.values < _lcl) & (_lcl > 0),
                            alpha=0.15, color="#1976D2", label="Below LCL")

            _adf = _anom_days_cache.get(_dt, _pdv2.DataFrame())
            if len(_adf) > 0 and "date" in _adf.columns:
                _anom_idx  = _pdv2.to_datetime(_adf["date"].values)
                _anom_vals = _daily.reindex(_anom_idx, method="nearest")
                ax.scatter(_anom_vals.index, _anom_vals.values,
                           color="#D32F2F", s=35, zorder=5, label=f"Anomalous days ({len(_adf)})")

            ax2 = ax.twinx()
            _viol_pos = _viol_daily[_viol_daily > 0]
            if len(_viol_pos) > 0:
                ax2.bar(_viol_pos.index, _viol_pos.values,
                        width=0.9, alpha=0.35, color="#D32F2F", label="Violation device-hrs/day")
            ax2.set_ylabel("Violation device-hrs / day", color="#D32F2F", fontsize=8)
            ax2.tick_params(axis="y", labelcolor="#D32F2F", labelsize=7)

            _anom_cnt = len(_adf)
            _viol_pct = (_anom_cnt / len(_daily) * 100) if len(_daily) > 0 else 0
            ax.set_title(f"SPC Fleet Control Chart (Option A) -- {_dt}  |  "
                         f"Anomalous days: {_anom_cnt} ({_viol_pct:.1f}%)",
                         fontsize=11, fontweight="bold")
            ax.set_ylabel("Daily mean event count")
            ax.set_xlabel("Date")
            _h1, _l1 = ax.get_legend_handles_labels()
            _h2, _l2 = ax2.get_legend_handles_labels()
            ax.legend(_h1+_h2, _l1+_l2, fontsize=7, ncol=3, loc="upper left")

        plt.tight_layout()
        plt.savefig("/tmp/ps4_spc_chart.png", dpi=100); plt.show()
        _ct = globals().get("COMMON_TAGS", {"ps":"PS4","target":"ensemble_anomaly_flag"})
        with mlflow.start_run(run_name="spc_control_chart") as _r:
            mlflow.set_tags({**_ct, "model_type": "visualization"})
            mlflow.log_artifact("/tmp/ps4_spc_chart.png")
    else:
        print("[INFO] spc_results empty or no hour_dt -- SPC chart skipped.")
else:
    print("[INFO] daily_event_count or DEV_COL not available -- SPC chart skipped.")

# -- Anomalous days table (per device type)
_SEP12 = "=" * 65
if _anom_days_cache:
    print(f"\n{_SEP12}")
    print("Anomalous Days -- Top 20 per Device Type")
    print("  violation_device_hours = (device x hour) rows with signal_spc=1 on that day")
    print("  avg_event_count        = mean transactions per device per day (busyness)")
    print(_SEP12)
    for _dt, _adf in _anom_days_cache.items():
        if len(_adf) == 0: continue
        _viol_col = next((c for c in ["violation_device_hours","violation_hours"] if c in _adf.columns), None)
        print(f"\n  {_dt}  --  {len(_adf)} anomalous days total")
        _show_cols = ["date"] + ([_viol_col] if _viol_col else []) + \
                     (["avg_event_count"] if "avg_event_count" in _adf.columns else [])
        _show = _adf[_show_cols].head(20).copy()
        _show["date"] = _show["date"].astype(str).str[:10]
        print(_show.to_string(index=False))

# -- Device-level breakdown: which devices drove violations on worst days
_dev_id_col = next((c for c in ["DEVICE_ID","device_id"] if c in df.columns), None)
_dev_nm_col = next((c for c in ["DEVICE_NAME","device_name"] if c in df.columns), None)
_fac_id_col = next((c for c in ["FACILITY_ID","facility_id","facid"] if c in df.columns), None)
_fac_nm_col = next((c for c in ["FACILITY_NAME","facility_name"] if c in df.columns), None)

if "signal_spc" in df.columns and _dev_id_col and DEV_COL in df.columns and "hour_dt" in df.columns:
    print(f"\n{_SEP12}")
    print("Top 5 Devices Driving SPC Violations (per device type)")
    print("  Total violation device-hours across full date range.")
    print(_SEP12)
    for _dt12 in DEVICE_TYPES:
        _mask12 = df[DEV_COL].str.strip().str.upper() == _dt12
        if _mask12.sum() == 0: continue
        _grp12  = (df[_mask12].groupby(_dev_id_col)["signal_spc"]
                              .agg(violation_device_hours="sum", total_hours="count",
                                   violation_rate="mean")
                              .sort_values("violation_device_hours", ascending=False)
                              .head(5).reset_index())
        if _dev_nm_col:
            _nm12 = df[[_dev_id_col,_dev_nm_col]].drop_duplicates().set_index(_dev_id_col)[_dev_nm_col]
            _grp12["DEVICE_NAME"] = _grp12[_dev_id_col].map(_nm12)
        _fleet_viol = int(df[_mask12]["signal_spc"].sum())
        _pct_top5   = _grp12["violation_device_hours"].sum() / _fleet_viol * 100 if _fleet_viol else 0
        print(f"\n  {_dt12}  ({_fleet_viol:,} total fleet violation device-hours)")
        print(_grp12.round(3).to_string(index=False))
        print(f"    Top 5 account for {_pct_top5:.1f}% of all {_dt12} violations")

# -- Facility-level breakdown: top 5 facilities by total violation device-hours
if "signal_spc" in df.columns and _fac_id_col and DEV_COL in df.columns:
    print(f"\n{_SEP12}")
    print("Top 5 Facilities Driving SPC Violations (per device type)")
    print(_SEP12)
    for _dt12 in DEVICE_TYPES:
        _mask12 = df[DEV_COL].str.strip().str.upper() == _dt12
        if _mask12.sum() == 0: continue
        _gcols12 = [_fac_id_col] + ([_fac_nm_col] if _fac_nm_col else [])
        _fgrp12  = (df[_mask12].groupby(_gcols12)
                               .agg(violation_device_hours=("signal_spc","sum"),
                                    total_device_hours=(_fac_id_col,"count"),
                                    violation_rate=("signal_spc","mean"),
                                    n_devices=(_dev_id_col,"nunique") if _dev_id_col else ("signal_spc","count"))
                               .sort_values("violation_device_hours", ascending=False)
                               .head(5).reset_index())
        print(f"\n  {_dt12}:")
        print(_fgrp12.round(3).to_string(index=False))

# -- Signal activation timeline (7-day rolling, daily grain)
_time_sigs = [s for s in SIGNAL_COLS + [TARGET] if s in df.columns]
if "hour_dt" in df.columns and len(_time_sigs):
    _ts = (_signal_ts_cache
           if _signal_ts_cache is not None and all(s in _signal_ts_cache.columns for s in _time_sigs)
           else df.set_index("hour_dt")[_time_sigs].sort_index()
                  .resample("1D").mean().rolling(7, min_periods=1).mean())
    fig, ax = plt.subplots(figsize=(18, 5))
    _COLORS = ["#1565C0","#E65100","#2E7D32","#6A1B9A","#00838F","#C62828","#F57F17"]
    for _idx, s in enumerate(_time_sigs):
        ax.plot(_ts.index, _ts[s], label=s,
                lw=2.5 if s == TARGET else 1.2,
                ls="-" if s == TARGET else "--",
                color=_COLORS[_idx % len(_COLORS)])
    ax.set_title("Signal activation rate -- 7-day rolling average", fontsize=11, fontweight="bold")
    ax.set_ylabel("Activation rate")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1%}"))
    ax.legend(ncol=2, fontsize=8); plt.tight_layout()
    plt.savefig("/tmp/ps4_signal_timeline.png", dpi=100); plt.show()
    _ct = globals().get("COMMON_TAGS", {"ps":"PS4","target":"ensemble_anomaly_flag"})
    with mlflow.start_run(run_name="signal_timeline") as _r:
        mlflow.set_tags({**_ct, "model_type": "visualization"})
        mlflow.log_artifact("/tmp/ps4_signal_timeline.png")

# -- SPC threshold table
if spc_results:
    print(f"\n{'─'*50}")
    print("SPC Thresholds")
    print(f"{'─'*50}")
    print(_pdv2.DataFrame(spc_results).T.round(2).to_string())
else:
    print("[INFO] SPC thresholds empty.")


In [ ]:
# ── CELL 13 : Signal 2 — Isolation Forest (PCA-reduced, per-device contamination) ─────
# PCA(95%) before IF: reduces 50+ features to ~15 components so IF trees
# sample the right dimensions instead of random noise columns.
# Per-device contamination: avoids one-size-fits-all 5% hardcoded rate.

IF_FEATURES = [c for c in FEATURE_COLS
               if any(k in c.lower() for k in
                      ["metric","response","percent_loss","kpi","rate","count"])]
IF_FEATURES = IF_FEATURES if IF_FEATURES else FEATURE_COLS[:15]

# ── Checkpoint guard (skip fit if model already saved) ────────────────────
import joblib as _jl13, os as _os13
from sklearn.decomposition import PCA as _PCA
_IF_CKPT   = os.path.join(CKPT_DIR, "if_model.joblib")
FORCE_IF   = False
_if_loaded = False

if not FORCE_IF and _os13.path.exists(_IF_CKPT):
    _ck13          = _jl13.load(_IF_CKPT)
    scaler         = _ck13["scaler"]
    _pca           = _ck13["pca"]
    iforest        = _ck13["iforest"]
    labels_train   = _ck13["labels_train"]
    labels_val     = _ck13["labels_val"]
    labels_test    = _ck13["labels_test"]
    _global_contam = _ck13.get("global_contam", float(np.clip(y_train.mean(), 0.01, 0.40)))
    _dev_contam    = _ck13.get("dev_contam", {})
    # Guard: IF_FEATURES can grow after checkpoint was saved
    # (e.g. Cell 11 Option A appends fleet_spc_rate which contains 'rate' -> included in IF_FEATURES).
    # If count changed, delete stale checkpoint and fall through to fresh refit.
    if scaler.n_features_in_ != len(IF_FEATURES):
        print(f"[WARN] Scaler expects {scaler.n_features_in_} features but "
              f"IF_FEATURES now has {len(IF_FEATURES)} — stale checkpoint deleted, refitting.")
        _os13.remove(_IF_CKPT)
    else:
        # Recompute PCA-space arrays (fast inference transforms) — needed for Cell 14 + df_if_score
        X_if_tr     = scaler.transform(train_df[IF_FEATURES].fillna(0).values)
        X_if_va     = scaler.transform(val_df[IF_FEATURES].fillna(0).values)
        X_if_te     = scaler.transform(test_df[IF_FEATURES].fillna(0).values)
        X_if_tr_pca = _pca.transform(X_if_tr)
        X_if_va_pca = _pca.transform(X_if_va)
        X_if_te_pca = _pca.transform(X_if_te)
        _if_loaded   = True
        print(f"[CHECKPOINT] IF loaded — {_pca.n_components_} PCA components, skipping fit")

if not _if_loaded:
    scaler      = StandardScaler()
    X_if_tr     = scaler.fit_transform(train_df[IF_FEATURES].fillna(0).values)
    X_if_va     = scaler.transform(val_df[IF_FEATURES].fillna(0).values)
    X_if_te     = scaler.transform(test_df[IF_FEATURES].fillna(0).values)

    _pca        = _PCA(n_components=0.95, random_state=42)
    X_if_tr_pca = _pca.fit_transform(X_if_tr)
    X_if_va_pca = _pca.transform(X_if_va)
    X_if_te_pca = _pca.transform(X_if_te)
    print(f"PCA: {len(IF_FEATURES)} features -> {_pca.n_components_} components (95% variance)")

    _global_contam = float(np.clip(y_train.mean(), 0.01, 0.40))
    if DEV_COL and DEV_COL in train_df.columns:
        _dev_contam = {}
        for _dt in DEVICE_TYPES:
            _mask = train_df[DEV_COL].str.strip().str.upper() == _dt
            _rate = float(train_df[_mask][TARGET].mean()) if _mask.sum() > 0 else _global_contam
            _dev_contam[_dt] = round(np.clip(_rate, 0.01, 0.40), 4)
        print(f"Per-device contamination: {_dev_contam}")
    else:
        _dev_contam = {}

    iforest = IsolationForest(
        n_estimators  = 200,
        contamination = _global_contam,
        max_samples   = "auto",
        random_state  = 42,
        n_jobs        = -1,
    )
    iforest.fit(X_if_tr_pca)
    labels_train = (iforest.predict(X_if_tr_pca) == -1).astype(int)
    labels_val   = (iforest.predict(X_if_va_pca) == -1).astype(int)
    labels_test  = (iforest.predict(X_if_te_pca) == -1).astype(int)

    _jl13.dump({
        "scaler":       scaler,
        "pca":          _pca,
        "iforest":      iforest,
        "labels_train": labels_train,
        "labels_val":   labels_val,
        "labels_test":  labels_test,
        "global_contam": _global_contam,
        "dev_contam":   _dev_contam,
    }, _IF_CKPT)
    print(f"[CHECKPOINT] IF saved -> {_IF_CKPT}")

# ── MLflow: write-back + logging (always runs whether loaded or fresh) ────
with mlflow.start_run(run_name="isolation_forest_signal2") as run:
    mlflow.set_tags({**COMMON_TAGS, "model_type": "isolation_forest", "signal": "S2"})

    import pandas as _pd4
    _if_flags = _pd4.Series(
        dict(list(zip(train_df.index, labels_train)) +
             list(zip(val_df.index,   labels_val))   +
             list(zip(test_df.index,  labels_test))),
        name="signal_isolation_forest")
    df["signal_isolation_forest"] = _if_flags.reindex(df.index).fillna(0).astype(int)

    # Recompute ensemble flag now that IF is real (not placeholder 0)
    df["signal_active_count"] = df[SIGNAL_COLS].sum(axis=1)
    df[TARGET] = (df["signal_active_count"] >= SIGNAL_THRESHOLD).astype(int)
    y_train = train_df.join(df[[TARGET]], rsuffix="_r")[TARGET].values
    y_val   = val_df.join(df[[TARGET]],   rsuffix="_r")[TARGET].values
    y_test  = test_df.join(df[[TARGET]],  rsuffix="_r")[TARGET].values
    y_cv    = np.concatenate([y_train, y_val])
    neg, pos = (y_train==0).sum(), max((y_train==1).sum(), 1)
    SCALE_POS_WEIGHT = round(neg / pos, 2)

    print(f"\nAfter PCA+IF signal update:")
    for s in SIGNAL_COLS:
        print(f"  {s}: {df[s].mean():.2%}")
    print(f"  ensemble_anomaly_flag: {df[TARGET].mean():.2%}")
    print(f"  scale_pos_weight updated: {SCALE_POS_WEIGHT}")

    mlflow.log_params({"n_estimators": 200, "contamination": round(_global_contam, 4),
                       "pca_components": int(_pca.n_components_),
                       "if_features_raw": len(IF_FEATURES)})
    if _dev_contam:
        mlflow.log_params({f"contamination_{k.lower()}": v for k, v in _dev_contam.items()})
    mlflow.log_metric("if_anomaly_rate_test", round(float(labels_test.mean()), 4))
    if len(labels_test) == len(y_test):
        from sklearn.metrics import f1_score as _f1s
        mlflow.log_metric("if_f1_vs_ensemble",
                          round(float(_f1s(y_test, labels_test, zero_division=0)), 4))

    joblib.dump(iforest, "/tmp/ps4_iforest.joblib")
    joblib.dump(scaler,  "/tmp/ps4_if_scaler.joblib")
    joblib.dump(_pca,    "/tmp/ps4_if_pca.joblib")
    mlflow.log_artifact("/tmp/ps4_iforest.joblib",   "isolation_forest")
    mlflow.log_artifact("/tmp/ps4_if_scaler.joblib", "isolation_forest")
    mlflow.log_artifact("/tmp/ps4_if_pca.joblib",    "isolation_forest")
    IF_RUN = run.info.run_id

    # IF score as numeric feature (decision_function — forward pass, fast even from checkpoint)
    df_if_score = _pd4.concat([
        _pd4.Series(iforest.decision_function(X_if_tr_pca), index=train_df.index, name="if_score"),
        _pd4.Series(iforest.decision_function(X_if_va_pca), index=val_df.index,   name="if_score"),
        _pd4.Series(iforest.decision_function(X_if_te_pca), index=test_df.index,  name="if_score"),
    ])
    if "if_score" not in df.columns:
        df = df.join(df_if_score)
    else:
        df["if_score"] = df_if_score
    if "if_score" not in FEATURE_COLS: FEATURE_COLS.append("if_score")
    X_train = train_df.join(df_if_score)[FEATURE_COLS].fillna(0).values
    X_val   = val_df.join(df_if_score)[FEATURE_COLS].fillna(0).values
    X_test  = test_df.join(df_if_score)[FEATURE_COLS].fillna(0).values
    X_cv    = np.vstack([X_train, X_val])

print(f"IF (PCA-reduced) done. Features: {len(FEATURE_COLS)}")


# -- Device-level IF breakdown (per device type)
_SEP13 = "=" * 65
_dev_id_col = next((c for c in ["DEVICE_ID","device_id"] if c in df.columns), None)
_dev_nm_col = next((c for c in ["DEVICE_NAME","device_name"] if c in df.columns), None)
_fac_id_col = next((c for c in ["FACILITY_ID","facility_id","facid"] if c in df.columns), None)
_fac_nm_col = next((c for c in ["FACILITY_NAME","facility_name"] if c in df.columns), None)

if DEV_COL in train_df.columns and _dev_id_col:
    import pandas as _pd4b
    # Build per-split label Series indexed like each split's df
    _if_lbl_tr = _pd4b.Series(labels_train.astype(int), index=train_df.index, name="if_flag")
    _if_lbl_va = _pd4b.Series(labels_val.astype(int),   index=val_df.index,   name="if_flag")
    _if_lbl_te = _pd4b.Series(labels_test.astype(int),  index=test_df.index,  name="if_flag")

    print(f"\n{_SEP13}")
    print("Isolation Forest -- Device-Type Anomaly Rate Summary")
    print("  if_flag = 1 means IF classified that device-hour as anomalous.")
    print(_SEP13)
    for _split_name, _split_df, _if_lbl in [
            ("Train", train_df, _if_lbl_tr),
            ("Val",   val_df,   _if_lbl_va),
            ("Test",  test_df,  _if_lbl_te)]:
        _tmp = _split_df[[DEV_COL]].copy()
        _tmp["if_flag"] = _if_lbl
        _rates = _tmp.groupby(DEV_COL)["if_flag"].mean()
        _parts = "  ".join(f"{dt}={_rates.get(dt, float('nan')):.1%}" for dt in DEVICE_TYPES)
        print(f"  {_split_name:6s}: {_parts}")

    # Top 5 most anomalous devices by IF flag rate (training set)
    print(f"\n{_SEP13}")
    print("Top 5 Devices with Highest IF Anomaly Rate (training set)")
    print("  if_flag_rate: fraction of hours IF flagged as anomalous")
    print("  Persistently high IF rate = device behavior consistently outlier-like")
    print(_SEP13)
    _tr_if = train_df[[DEV_COL, _dev_id_col]].copy()
    _tr_if["if_flag"] = _if_lbl_tr
    for _dt13 in DEVICE_TYPES:
        _mask13 = _tr_if[DEV_COL].str.strip().str.upper() == _dt13
        if _mask13.sum() == 0: continue
        _grp13 = (_tr_if[_mask13].groupby(_dev_id_col)["if_flag"]
                                  .agg(if_flag_rate="mean", n_hours="count")
                                  .sort_values("if_flag_rate", ascending=False)
                                  .head(5).reset_index())
        if _dev_nm_col:
            _nm13 = df[[_dev_id_col,_dev_nm_col]].drop_duplicates().set_index(_dev_id_col)[_dev_nm_col]
            _grp13["DEVICE_NAME"] = _grp13[_dev_id_col].map(_nm13)
        print(f"\n  {_dt13}:")
        print(_grp13.round(3).to_string(index=False))

    # Top 5 most anomalous facilities by IF flag rate (training set)
    if _fac_id_col and _fac_id_col in train_df.columns:
        print(f"\n{_SEP13}")
        print("Top 5 Facilities with Highest IF Anomaly Rate (training set)")
        print(_SEP13)
        _gcols13 = [_fac_id_col] + ([_fac_nm_col] if _fac_nm_col and _fac_nm_col in train_df.columns else [])
        _tr_fac13 = train_df[_gcols13 + [DEV_COL, _dev_id_col]].copy()
        _tr_fac13["if_flag"] = _if_lbl_tr
        for _dt13 in DEVICE_TYPES:
            _mask13 = _tr_fac13[DEV_COL].str.strip().str.upper() == _dt13
            if _mask13.sum() == 0: continue
            _fg13 = (_tr_fac13[_mask13].groupby(_gcols13)
                                        .agg(if_flag_rate=("if_flag","mean"),
                                             n_devices=(_dev_id_col,"nunique"),
                                             n_hours=(_dev_id_col,"count"))
                                        .sort_values("if_flag_rate", ascending=False)
                                        .head(5).reset_index())
            print(f"\n  {_dt13}:")
            print(_fg13.round(3).to_string(index=False))


In [ ]:
# -- CELL 14 : PCA + IF diagnostic charts + device/facility top-5
import pandas as _pdv3
from sklearn.metrics import precision_recall_curve, roc_curve, auc as _auc_fn

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# -- 1. PCA explained variance curve
_cum_var = np.cumsum(_pca.explained_variance_ratio_)
axes[0,0].plot(range(1, len(_cum_var)+1), _cum_var, marker="o", ms=4, color="#1565C0")
axes[0,0].axhline(0.95, color="red", ls="--", lw=1, label="95% threshold")
axes[0,0].axvline(_pca.n_components_, color="green", ls="--", lw=1,
                  label=f"{_pca.n_components_} components selected")
axes[0,0].fill_between(range(1, _pca.n_components_+1), _cum_var[:_pca.n_components_],
                       alpha=0.2, color="blue")
axes[0,0].set_title("PCA -- cumulative explained variance", fontsize=10, fontweight="bold")
axes[0,0].set_xlabel("# Principal Components"); axes[0,0].set_ylabel("Variance explained")
axes[0,0].legend(fontsize=8)

# -- 2. IF anomaly score distribution on training set
_tr_scores = iforest.decision_function(X_if_tr_pca)
axes[0,1].hist(_tr_scores[y_train==0], bins=50, alpha=0.6, color="#1565C0",
               label="Normal (train)", density=True)
axes[0,1].hist(_tr_scores[y_train==1], bins=50, alpha=0.6, color="#E53935",
               label="Anomaly (train)", density=True)
axes[0,1].axvline(0, color="black", ls="--", lw=1.2, label="Decision boundary")
axes[0,1].set_title("IF score distribution (train)", fontsize=10, fontweight="bold")
axes[0,1].set_xlabel("Decision score (lower = more anomalous)")
axes[0,1].legend(fontsize=8)

# -- 3. IF score on test set
_te_scores = iforest.decision_function(X_if_te_pca)
axes[0,2].hist(_te_scores[y_test==0], bins=50, alpha=0.6, color="#43A047",
               label="Normal (test)", density=True)
axes[0,2].hist(_te_scores[y_test==1], bins=50, alpha=0.6, color="#FF6F00",
               label="Anomaly (test)", density=True)
axes[0,2].axvline(0, color="black", ls="--", lw=1.2, label="Decision boundary")
axes[0,2].set_title("IF score distribution (test)", fontsize=10, fontweight="bold")
axes[0,2].set_xlabel("Decision score (lower = more anomalous)")
axes[0,2].legend(fontsize=8)

# -- 4. Precision-Recall curve (IF score vs ensemble flag)
_pr_score = -_te_scores   # negate so higher = more anomalous
_prec_c, _rec_c, _ = precision_recall_curve(y_test, _pr_score)
_ap_if = float(np.trapezoid(_prec_c, _rec_c))
axes[1,0].plot(_rec_c, _prec_c, color="#1565C0", lw=2, label=f"IF (AP={_ap_if:.3f})")
axes[1,0].axhline(y_test.mean(), color="gray", ls="--", lw=1,
                  label=f"Baseline ({y_test.mean():.3f})")
axes[1,0].set_title("Precision-Recall -- IF vs ensemble_anomaly_flag (test)",
                    fontsize=10, fontweight="bold")
axes[1,0].set_xlabel("Recall"); axes[1,0].set_ylabel("Precision")
axes[1,0].legend(fontsize=8)

# -- 5. ROC curve
_fpr, _tpr, _ = roc_curve(y_test, _pr_score)
_roc_auc = _auc_fn(_fpr, _tpr)
axes[1,1].plot(_fpr, _tpr, color="#7B1FA2", lw=2, label=f"IF (AUC={_roc_auc:.3f})")
axes[1,1].plot([0,1], [0,1], color="gray", ls="--", lw=1, label="Random")
axes[1,1].set_title("ROC curve -- IF score (test)", fontsize=10, fontweight="bold")
axes[1,1].set_xlabel("False Positive Rate"); axes[1,1].set_ylabel("True Positive Rate")
axes[1,1].legend(fontsize=8)

# -- 6. Per-device contamination bar
if _dev_contam:
    _dc = _pdv3.Series(_dev_contam)
    _dc.plot(kind="bar", ax=axes[1,2], color="#00897B", edgecolor="white")
    for bar, v in zip(axes[1,2].patches, _dc.values):
        axes[1,2].text(bar.get_x()+bar.get_width()/2, v+0.001, f"{v:.2%}",
                       ha="center", fontsize=9)
    axes[1,2].set_title("Per-device IF contamination rate", fontsize=10, fontweight="bold")
    axes[1,2].set_ylabel("Contamination"); axes[1,2].tick_params(axis="x", rotation=0)
else:
    axes[1,2].text(0.5, 0.5, "No per-device contamination data",
                   ha="center", va="center", transform=axes[1,2].transAxes)

plt.tight_layout()
plt.savefig("/tmp/ps4_pca_if_charts.png", dpi=100); plt.show()
with mlflow.start_run(run_name="pca_if_diagnostics") as _r:
    mlflow.set_tags({**COMMON_TAGS, "model_type": "visualization"})
    mlflow.log_artifact("/tmp/ps4_pca_if_charts.png")

# -- Top-10 feature loadings table
print("\n-- Top 10 IF features (absolute PCA loadings across first 3 components) --")
_load = _pdv3.DataFrame(
    np.abs(_pca.components_[:3]).T,
    index=IF_FEATURES,
    columns=["PC1", "PC2", "PC3"])
_load["max_load"] = _load.max(axis=1)
print(_load.sort_values("max_load", ascending=False).head(10).round(4).to_string())

# -- Top 5 most anomalous devices by IF score (test set)
# Lower decision_function score = further from "normal" = more anomalous
_dev_id_col = next((c for c in ["DEVICE_ID","device_id"] if c in df.columns), None)
_dev_nm_col = next((c for c in ["DEVICE_NAME","device_name"] if c in df.columns), None)
_fac_id_col = next((c for c in ["FACILITY_ID","facility_id","facid"] if c in df.columns), None)
_fac_nm_col = next((c for c in ["FACILITY_NAME","facility_name"] if c in df.columns), None)

if _dev_id_col and DEV_COL:
    _te_df = test_df.copy()
    _te_df["if_score"]    = _te_scores   # lower = more anomalous
    _te_df["if_anomaly"]  = (labels_test == 1).astype(int)

    _SEP14 = "=" * 65
    print(f"\n{_SEP14}")
    print("Top 5 Most Anomalous Devices by IF Score (test set)")
    print("  if_score: lower = more anomalous (further from normal cluster)")
    print("  if_anomaly_rate: fraction of device's test hours flagged by IF")
    print(_SEP14)

    for _dt14 in DEVICE_TYPES:
        _mask14 = _te_df[DEV_COL].str.strip().str.upper() == _dt14
        if _mask14.sum() == 0: continue
        _sub14  = _te_df[_mask14].copy()
        _grp14  = (_sub14.groupby(_dev_id_col)
                         .agg(if_score_mean=("if_score","mean"),
                              if_anomaly_rate=("if_anomaly","mean"),
                              n_hours=(_dev_id_col,"count"))
                         .sort_values("if_score_mean").head(5).reset_index())
        if _dev_nm_col:
            _nm14 = (df[[_dev_id_col,_dev_nm_col]].drop_duplicates()
                        .set_index(_dev_id_col)[_dev_nm_col])
            _grp14["DEVICE_NAME"] = _grp14[_dev_id_col].map(_nm14)
        print(f"\n  {_dt14}:")
        print(_grp14.round(4).to_string(index=False))

    # -- Top 5 most anomalous facilities by IF anomaly rate (test set)
    if _fac_id_col:
        print(f"\n{_SEP14}")
        print("Top 5 Most Anomalous Facilities by IF Anomaly Rate (test set)")
        print(_SEP14)
        for _dt14 in DEVICE_TYPES:
            _mask14 = _te_df[DEV_COL].str.strip().str.upper() == _dt14
            if _mask14.sum() == 0: continue
            _sub14  = _te_df[_mask14].copy()
            _gcols  = [_fac_id_col] + ([_fac_nm_col] if _fac_nm_col else [])
            _fg14   = (_sub14.groupby(_gcols)
                             .agg(if_anomaly_rate=("if_anomaly","mean"),
                                  if_score_mean=("if_score","mean"),
                                  n_devices=(_dev_id_col,"nunique") if _dev_id_col else ("if_anomaly","count"))
                             .sort_values("if_anomaly_rate", ascending=False)
                             .head(5).reset_index())
            print(f"\n  {_dt14}:")
            print(_fg14.round(4).to_string(index=False))


In [ ]:
# ── CELL 15 : Station-level aggregation — descriptive analytics + charts ──────
# Station-level: if >50% of devices at a station are anomalous simultaneously
#   → infrastructure fault, not individual device fault.
# Analytics: anomaly rate distribution, top stations, time trends, feature stats.
# Paired device model: wide-format XGBoost to distinguish device vs station faults.

import pandas as _pd15s, numpy as _np15s, matplotlib.pyplot as _plt15s, seaborn as _sns15s

STATION_COL = next((c for c in df.columns
                    if any(k in c.lower() for k in
                           ["station","location","site","place","stop","facility"])), None)

_SEP = "=" * 65

# ── 0. Descriptive overview of the dataset ─────────────────────────────────────
print(_SEP)
print("PS4  –  Dataset Overview (before station aggregation)")
print(_SEP)
print(f"  Total rows       : {len(df):,}")
print(f"  Date range       : {df['hour_dt'].min()} → {df['hour_dt'].max()}")
print(f"  Target ({TARGET}) : {df[TARGET].sum():,} anomaly hours  ({df[TARGET].mean():.2%} rate)")
if DEV_COL and DEV_COL in df.columns:
    print(f"  Device types     : {sorted(df[DEV_COL].str.strip().str.upper().unique())}")
print(f"  Feature cols     : {len(FEATURE_COLS)}")
print(f"  Train / Val / Test rows: {len(train_df):,} / {len(val_df):,} / {len(test_df):,}")

# ── 0a. Feature distribution summary ──────────────────────────────────────────
print(f"\n{_SEP}")
print("Feature Distribution Summary (training set, top 10 features by variance)")
print(_SEP)
# Force numeric — X_train may have object dtype when built from mixed-type pipeline
_tr_arr  = _np15s.array(X_train, dtype=float)
_tr_feat = _pd15s.DataFrame(_tr_arr, columns=FEATURE_COLS)
_var_top = _tr_feat.var().nlargest(10).index.tolist()
if _var_top:
    print(_tr_feat[_var_top].describe().round(4).to_string())
else:
    print(f"  [{len(FEATURE_COLS)} feature columns; variance summary skipped — all constant]")

# ── 0b. Anomaly rate by device type ───────────────────────────────────────────
if DEV_COL and DEV_COL in df.columns:
    print(f"\n{_SEP}")
    print("Anomaly Rate by Device Type")
    print(_SEP)
    _dev_anom = (df.groupby(df[DEV_COL].str.strip().str.upper())[TARGET]
                   .agg(anomaly_rate="mean", total_hours="count", anomaly_hours="sum")
                   .reset_index().rename(columns={DEV_COL: "device_type"})
                   .sort_values("anomaly_rate", ascending=False))
    print(_dev_anom.round(4).to_string(index=False))

    # Chart: anomaly rate by device type
    _fig0, _ax0 = _plt15s.subplots(figsize=(8, 4))
    _ax0.bar(_dev_anom["device_type"], _dev_anom["anomaly_rate"] * 100,
             color="#1565C0", edgecolor="white")
    _ax0.set_title("Anomaly Rate (%) by Device Type", fontsize=12, fontweight="bold")
    _ax0.set_xlabel("Device Type"); _ax0.set_ylabel("Anomaly Rate (%)")
    for _p in _ax0.patches:
        _ax0.annotate(f"{_p.get_height():.1f}%",
                      (_p.get_x() + _p.get_width()/2, _p.get_height()),
                      ha="center", va="bottom", fontsize=9)
    _fig0.tight_layout(); _plt15s.show()
    with mlflow.start_run(run_name="anomaly_rate_by_device") as _r0:
        mlflow.set_tags({**COMMON_TAGS, "model_type": "visualization"})
        _fig0.savefig("/tmp/ps4_device_anomaly_rate.png", dpi=100)
        mlflow.log_artifact("/tmp/ps4_device_anomaly_rate.png")

# ── 0c. Anomaly trend over time ────────────────────────────────────────────────
print(f"\n{_SEP}")
print("Monthly Anomaly Rate Trend")
print(_SEP)
_df_ts = df.copy()
_df_ts["month"] = _pd15s.to_datetime(_df_ts["hour_dt"]).dt.to_period("M").astype(str)
_monthly = (_df_ts.groupby("month")[TARGET]
              .agg(anomaly_rate="mean", total_hours="count").reset_index())
print(_monthly.round(4).to_string(index=False))

_fig1, _ax1 = _plt15s.subplots(figsize=(12, 4))
_ax1.plot(_monthly["month"], _monthly["anomaly_rate"] * 100,
          marker="o", color="#1565C0", linewidth=2, markersize=5)
_ax1.fill_between(range(len(_monthly)), _monthly["anomaly_rate"] * 100,
                  alpha=0.15, color="#1565C0")
_ax1.set_xticks(range(len(_monthly)))
_ax1.set_xticklabels(_monthly["month"], rotation=45, ha="right", fontsize=8)
_ax1.set_title("Monthly Anomaly Rate Trend", fontsize=12, fontweight="bold")
_ax1.set_xlabel("Month"); _ax1.set_ylabel("Anomaly Rate (%)")
_ax1.axvline(x=_monthly[_monthly["month"] < "2025-10"].shape[0] - 0.5,
             color="orange", linestyle="--", lw=1.5, label="Train/Val split")
_ax1.axvline(x=_monthly[_monthly["month"] < "2026-01"].shape[0] - 0.5,
             color="red", linestyle="--", lw=1.5, label="Val/Test split")
_ax1.legend(fontsize=9); _fig1.tight_layout(); _plt15s.show()
with mlflow.start_run(run_name="monthly_anomaly_trend") as _r1:
    mlflow.set_tags({**COMMON_TAGS, "model_type": "visualization"})
    _fig1.savefig("/tmp/ps4_monthly_trend.png", dpi=100)
    mlflow.log_artifact("/tmp/ps4_monthly_trend.png")

# ── 0d. Feature correlation with target ────────────────────────────────────────
print(f"\n{_SEP}")
print("Top 15 Features Correlated with Anomaly Target (training set)")
print(_SEP)
_corr_df  = _pd15s.DataFrame(X_train, columns=FEATURE_COLS)
_corr_df["_target"] = y_train
_corr = _corr_df.corr()["_target"].drop("_target").abs().sort_values(ascending=False).head(15)
print(_corr.round(4).to_string())

_fig2, _ax2 = _plt15s.subplots(figsize=(8, 5))
_corr.sort_values().plot(kind="barh", ax=_ax2, color="#7B1FA2", edgecolor="white")
_ax2.set_title("Top 15 Features — Absolute Correlation with Anomaly Target",
               fontsize=11, fontweight="bold")
_ax2.set_xlabel("|Pearson correlation|"); _fig2.tight_layout(); _plt15s.show()
with mlflow.start_run(run_name="feature_target_correlation") as _r2:
    mlflow.set_tags({**COMMON_TAGS, "model_type": "visualization"})
    _fig2.savefig("/tmp/ps4_feature_corr.png", dpi=100)
    mlflow.log_artifact("/tmp/ps4_feature_corr.png")

# ── 1. Station-level aggregation ──────────────────────────────────────────────
if STATION_COL is None:
    print(f"\n{_SEP}")
    print("[INFO] No station column found in Gold table.")
    print("       Keywords searched: station, location, site, place, stop, facility")
    print("       Add a station→device mapping to Gold ETL to enable station analysis.")
    print("       station_anomaly_rate = n_anomalous_devices / n_total_devices")
    print("       station_fault        = station_anomaly_rate > 0.5")
    STATION_ANOMALY_AVAILABLE = False
    X_train = train_df[FEATURE_COLS].fillna(0).values
    X_val   = val_df[FEATURE_COLS].fillna(0).values
    X_test  = test_df[FEATURE_COLS].fillna(0).values
    X_cv    = _np15s.vstack([X_train, X_val])
    print(f"  X arrays rebuilt from original splits. Features: {len(FEATURE_COLS)}")
else:
    print(f"\n{_SEP}")
    print(f"Station column found: {STATION_COL}")
    print(_SEP)
    import pandas as _pd6

    _station_hourly = (df.groupby([STATION_COL, "hour_dt"])
                         .agg(n_devices   = ("DEVICE_ID", "nunique"),
                              n_anomalous = (TARGET, "sum"))
                         .reset_index())
    _station_hourly["station_anomaly_rate"] = (
        _station_hourly["n_anomalous"] / _station_hourly["n_devices"].clip(lower=1))
    _station_hourly["station_fault"] = (
        _station_hourly["station_anomaly_rate"] > 0.5).astype(int)

    df = df.drop(columns=[c for c in ["station_anomaly_rate","station_fault"] if c in df.columns])
    df = df.merge(_station_hourly[[STATION_COL,"hour_dt","station_anomaly_rate","station_fault"]],
                  on=[STATION_COL,"hour_dt"], how="left")
    df["station_anomaly_rate"] = df["station_anomaly_rate"].fillna(0)
    df["station_fault"]        = df["station_fault"].fillna(0).astype(int)

    for _f in ["station_anomaly_rate","station_fault"]:
        if _f not in FEATURE_COLS: FEATURE_COLS.append(_f)

    _sf_rate = _station_hourly["station_fault"].mean()
    print(f"Station-level fault rate : {_sf_rate:.2%}")
    print(f"Total stations           : {df[STATION_COL].nunique():,}")
    print(f"Total station-hours      : {len(_station_hourly):,}")

    # ── 1a. Station anomaly rate summary ──────────────────────────────────────
    _st_summary = (_station_hourly.groupby(STATION_COL)["station_anomaly_rate"]
                     .agg(mean_anomaly_rate="mean", max_anomaly_rate="max",
                          pct_fault_hours=lambda x: (x > 0.5).mean(),
                          total_hours="count")
                     .reset_index()
                     .sort_values("mean_anomaly_rate", ascending=False))

    print(f"\n── Top 10 Stations by Mean Anomaly Rate ──")
    print(_st_summary.head(10).round(4).to_string(index=False))

    # Chart: top 15 stations by anomaly rate
    _top15_st = _st_summary.head(15)
    _fig3, _ax3 = _plt15s.subplots(figsize=(10, 5))
    _bars = _ax3.barh(_top15_st[STATION_COL].astype(str),
                      _top15_st["mean_anomaly_rate"] * 100,
                      color="#E53935", edgecolor="white")
    _ax3.set_title("Top 15 Stations — Mean Anomaly Rate (%)", fontsize=12, fontweight="bold")
    _ax3.set_xlabel("Mean Anomaly Rate (%)"); _ax3.invert_yaxis()
    for _b in _bars:
        _ax3.annotate(f"{_b.get_width():.1f}%",
                      (_b.get_width(), _b.get_y() + _b.get_height()/2),
                      ha="left", va="center", fontsize=8)
    _fig3.tight_layout(); _plt15s.show()
    with mlflow.start_run(run_name="top_stations_anomaly_rate") as _r3:
        mlflow.set_tags({**COMMON_TAGS, "model_type": "visualization"})
        _fig3.savefig("/tmp/ps4_top_stations.png", dpi=100)
        mlflow.log_artifact("/tmp/ps4_top_stations.png")

    # Chart: distribution of station anomaly rates
    _fig4, (_ax4a, _ax4b) = _plt15s.subplots(1, 2, figsize=(12, 4))
    _ax4a.hist(_st_summary["mean_anomaly_rate"] * 100, bins=30,
               color="#1565C0", edgecolor="white", alpha=0.8)
    _ax4a.set_title("Distribution of Station Anomaly Rates", fontsize=11, fontweight="bold")
    _ax4a.set_xlabel("Mean Anomaly Rate (%)"); _ax4a.set_ylabel("Number of Stations")
    _ax4a.axvline(x=50, color="red", linestyle="--", lw=1.5, label="Fault threshold (50%)")
    _ax4a.legend(fontsize=8)

    _ax4b.scatter(_st_summary["total_hours"], _st_summary["mean_anomaly_rate"] * 100,
                  alpha=0.5, color="#7B1FA2", s=20)
    _ax4b.set_title("Station Hours vs Anomaly Rate", fontsize=11, fontweight="bold")
    _ax4b.set_xlabel("Total Hours Monitored"); _ax4b.set_ylabel("Mean Anomaly Rate (%)")
    _fig4.tight_layout(); _plt15s.show()
    with mlflow.start_run(run_name="station_anomaly_distribution") as _r4:
        mlflow.set_tags({**COMMON_TAGS, "model_type": "visualization"})
        _fig4.savefig("/tmp/ps4_station_dist.png", dpi=100)
        mlflow.log_artifact("/tmp/ps4_station_dist.png")

    # ── 1b. Paired device model ────────────────────────────────────────────────
    if DEV_COL:
        _wide = (df[[STATION_COL, "hour_dt", DEV_COL, TARGET] + FEATURE_COLS[:10]]
                   .pivot_table(index=[STATION_COL,"hour_dt"],
                                columns=DEV_COL, values=FEATURE_COLS[:10],
                                aggfunc="mean"))
        _wide.columns = [f"{col[0]}_{col[1]}" for col in _wide.columns]
        _wide["station_fault"] = (df.groupby([STATION_COL,"hour_dt"])[TARGET].max()
                                    .reset_index(drop=True))
        _wide = _wide.fillna(0).reset_index(drop=True)
        _Xp = _wide.drop(columns=["station_fault"]).values
        _yp = _wide["station_fault"].values

        if len(_Xp) > 200:
            _n_tr = int(len(_Xp) * 0.7)
            _xgb_pair = XGBClassifier(
                n_estimators=200, max_depth=4, tree_method="hist",
                random_state=42, n_jobs=-1,
                scale_pos_weight=round((_yp==0).sum()/max((_yp==1).sum(),1),2))
            _xgb_pair.fit(_Xp[:_n_tr], _yp[:_n_tr])
            from sklearn.metrics import roc_auc_score as _ras2
            _pair_auc = _ras2(_yp[_n_tr:], _xgb_pair.predict_proba(_Xp[_n_tr:])[:,1])
            print(f"\nPaired device XGBoost  station_fault AUC = {_pair_auc:.4f}")
            with mlflow.start_run(run_name="paired_device_xgb") as _run:
                mlflow.set_tags({**COMMON_TAGS,"model_type":"paired_device_supervised"})
                mlflow.log_params({"n_station_pairs": len(_Xp),
                                   "paired_features": _Xp.shape[1]})
                mlflow.log_metric("station_fault_auc", round(_pair_auc, 4))
        else:
            print("[SKIP] Too few station-hour pairs for paired device model")

    # ── 1c. Rebuild X arrays with station features ─────────────────────────────
    _join_cols = list(
        {c for c in FEATURE_COLS if c not in train_df.columns and c in df.columns}
        | {"station_anomaly_rate", "station_fault"}
    )
    X_train = train_df.join(df[_join_cols])[FEATURE_COLS].fillna(0).values
    X_val   = val_df.join(df[_join_cols])[FEATURE_COLS].fillna(0).values
    X_test  = test_df.join(df[_join_cols])[FEATURE_COLS].fillna(0).values
    X_cv    = _np15s.vstack([X_train, X_val])
    STATION_ANOMALY_AVAILABLE = True
    print(f"\nStation features added. Total features: {len(FEATURE_COLS)}")
    print(f"X_train: {X_train.shape}  X_val: {X_val.shape}  X_test: {X_test.shape}")


In [ ]:
# ── CELL 16 : Summary ──────────────────────────────────────────────────
print("="*62)
print("PS4 — Hourly Anomaly Detection — Complete")
print("="*62)
print(f"Device column : {DEV_COL}")
print(f"Signal cols   : {SIGNAL_COLS}")
print(f"Monitor freq  : every 6h (hourly grain)")
print()
print("Signal activation rates (global):")
for s in SIGNAL_COLS:
    rate = df[s].mean() if s in df.columns else 0
    print(f"  {s:35s}: {rate:.2%}")
print()
print(f"  Ensemble anomaly rate : {df[TARGET].mean():.2%}")
print(f"  Total anomaly hours   : {df[TARGET].sum():,}")
print(f"  Total device-hours    : {len(df):,}")
print()
print(f"MLflow experiment : {EXPERIMENT_NAME}")


In [ ]:
# ── CELL 17 : Summary dashboard ──────────────────────────────────────────────────
import pandas as _pdv6

print("=" * 70)
print("PS4 — Hourly Anomaly Detection — Run Summary")
print("=" * 70)

# ── 1. Signal health table ─────────────────────────────────────────────────
print("\n[1] Signal Health")
_rows = [{"Signal": s,
          "Rate": f"{df[s].mean():.2%}",
          "Status": "ACTIVE" if df[s].mean() > 0.001 else "DORMANT"}
         for s in SIGNAL_COLS + [TARGET] if s in df.columns]
print(_pdv6.DataFrame(_rows).to_string(index=False))

# ── 2. Feature dimension table ─────────────────────────────────────────────
print(f"\n[2] Feature Space")
print(f"   Raw features in FEATURE_COLS : {len(FEATURE_COLS)}")
if "_pca" in dir():
    print(f"   IF features (raw)            : {len(IF_FEATURES)}")
    print(f"   PCA components (95% var)     : {_pca.n_components_}")

# ── 3. Train / Val / Test size ─────────────────────────────────────────────
print(f"\n[3] Data splits")
_sp = _pdv6.DataFrame([
    {"Split":"Train","Rows":len(X_train),"Anomaly_Rate":f"{y_train.mean():.2%}",
     "Date_Range":f"{train_df['hour_dt'].min().date()} → {train_df['hour_dt'].max().date()}"},
    {"Split":"Val",  "Rows":len(X_val),  "Anomaly_Rate":f"{y_val.mean():.2%}",
     "Date_Range":f"{val_df['hour_dt'].min().date()} → {val_df['hour_dt'].max().date()}"},
    {"Split":"Test", "Rows":len(X_test), "Anomaly_Rate":f"{y_test.mean():.2%}",
     "Date_Range":f"{test_df['hour_dt'].min().date()} → {test_df['hour_dt'].max().date()}"},
])
print(_sp.to_string(index=False))

# ── 4. Per-device champion table ──────────────────────────────────────────
if "PER_DEVICE_RESULTS" in dir() and PER_DEVICE_RESULTS:
    print("\n[4] Per-Device Champions")
    _champ_rows = []
    for _dt, _r in PER_DEVICE_RESULTS.items():
        _champ_rows.append({
            "Device": _dt,
            "Champion": _r.get("best_model_name", "?"),
            "Test_AUC": _r.get("test_auc", "?"),
            "Test_AP":  _r.get("test_ap",  "?"),
            "scale_pos_weight": _r.get("spw", "?"),
        })
    print(_pdv6.DataFrame(_champ_rows).to_string(index=False))

# ── 5. Dormant signal warning ─────────────────────────────────────────────
_dormant = [s for s in SIGNAL_COLS if s in df.columns and df[s].mean() < 0.001]
if _dormant:
    print(f"\n[WARN] Dormant signals (0% activation — check Gold ETL columns):")
    for s in _dormant:
        print(f"   {s}")

print("\n" + "=" * 70)


In [ ]:
# ── CELL 18 : Anomaly rate forecasting — Prophet ──────────────────────────────────────────────────
# Forecasts daily anomaly rate per device type for the next 7 days.
# Useful for maintenance scheduling: "GATE anomaly rate expected to peak Thu-Fri".
# Prophet decomposes trend + weekly seasonality + holiday effects.

try:
    from prophet import Prophet as _Prophet
    import pandas as _pd8
    _PROPHET_OK = True
except ImportError:
    print("[SKIP] Prophet not installed — run: pip install prophet")
    _PROPHET_OK = False

if _PROPHET_OK:
    _forecast_results = {}
    _dev_groups = [(dt, df[df[DEV_COL].str.strip().str.upper() == dt])
                   if DEV_COL else ("ALL", df)
                   for dt in (DEVICE_TYPES if DEV_COL else ["ALL"])]

    _fig, _axes = plt.subplots(len(_dev_groups), 1,
                                figsize=(12, 4 * len(_dev_groups)))
    if len(_dev_groups) == 1: _axes = [_axes]

    for _ax, (_dt, _ddf) in zip(_axes, _dev_groups):
        if len(_ddf) < 48:   # need at least 2 days of hourly data
            _ax.set_title(f"{_dt}: insufficient data"); continue

        # Daily anomaly rate time series
        _daily = (_ddf.assign(ds=_ddf["hour_dt"].dt.floor("D"))
                     .groupby("ds")[TARGET].mean()
                     .reset_index()
                     .rename(columns={TARGET: "y"}))
        _daily = _daily[_daily["y"].notna()]

        _m = _Prophet(yearly_seasonality=True, weekly_seasonality=True,
                      daily_seasonality=False, changepoint_prior_scale=0.05)
        _m.fit(_daily)

        _future   = _m.make_future_dataframe(periods=7)
        _forecast = _m.predict(_future)
        _forecast_results[_dt] = _forecast[["ds","yhat","yhat_lower","yhat_upper"]].tail(14)

        # Plot
        _hist = _daily.tail(60)
        _pred = _forecast.tail(7)
        _ax.plot(_hist["ds"], _hist["y"], "b-", lw=1.5, label="Historical")
        _ax.plot(_pred["ds"], _pred["yhat"], "r--", lw=2, label="Forecast")
        _ax.fill_between(_pred["ds"], _pred["yhat_lower"], _pred["yhat_upper"],
                         alpha=0.2, color="red", label="95% CI")
        _ax.set_title(f"{_dt} — 7-day anomaly rate forecast")
        _ax.set_ylabel("Anomaly rate"); _ax.legend(fontsize=8)
        _ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))

        with mlflow.start_run(run_name=f"prophet_forecast_{_dt.lower()}") as _run:
            mlflow.set_tags({**COMMON_TAGS, "model_type":"prophet_forecast",
                             "device_type":_dt})
            mlflow.log_param("forecast_horizon_days", 7)
            mlflow.log_metric("mean_daily_anomaly_rate",
                              round(float(_daily["y"].mean()), 4))

    plt.tight_layout()
    plt.savefig("/tmp/ps4_forecast.png", dpi=100); plt.show()
    with mlflow.start_run(run_name="anomaly_forecast_summary") as _run:
        mlflow.set_tags({**COMMON_TAGS, "model_type":"prophet_forecast"})
        mlflow.log_artifact("/tmp/ps4_forecast.png")
    print("Prophet forecasting done.")


In [ ]:
# ── FINAL CELL : Export notebook with all outputs to HTML ───────────────────
# Strategy:
#   1. Auto-discover notebook path on EFS
#   2. Force-save via Jupyter server REST API (flushes kernel outputs to disk)
#   3. Run nbconvert --to html
#   4. If outputs still missing, retry with --execute
import subprocess, os, datetime, time, re
try:
    import requests as _req
except ImportError:
    _req = None

_ts = datetime.datetime.now().strftime('%Y%m%d_%H%M')

# -- Step 1: locate the notebook file
_nb_path = None
_search_names = ['PS4_SageMaker_MLflow_FeatureStore.ipynb', 'PS4.ipynb']
for _name in _search_names:
    _r = subprocess.run(
        ['find', '/home/sagemaker-user', '-name', _name,
         '-not', '-path', '*/.ipynb_checkpoints/*'],
        capture_output=True, text=True, timeout=15
    )
    _hits = [p for p in _r.stdout.strip().splitlines() if os.path.exists(p)]
    if _hits:
        _nb_path = _hits[0]
        break
if not _nb_path:
    for _c in [os.path.join(os.getcwd(), _search_names[0]),
               '/home/sagemaker-user/PS4/PS4.ipynb']:
        if os.path.exists(_c):
            _nb_path = _c; break
if not _nb_path:
    print('[HTML] Notebook not found. Run: !find /home/sagemaker-user -name "*.ipynb" | grep -i ps4')
    raise SystemExit(1)
print(f'[HTML] Notebook: {_nb_path}')

# -- Step 2: force-save via Jupyter server API so outputs are in the file
_saved = False
if _req is not None:
    try:
        _srv = subprocess.run(['jupyter', 'server', 'list'],
                              capture_output=True, text=True, timeout=10)
        _m = re.search(r'http://[^\s]+', _srv.stdout + _srv.stderr)
        if _m:
            _srv_url = _m.group(0)
            _tok_m   = re.search(r'token=([a-f0-9]+)', _srv_url)
            _token   = _tok_m.group(1) if _tok_m else ''
            _base    = _srv_url.split('?')[0].rstrip('/')
            _rel     = _nb_path.replace('/home/sagemaker-user/', '', 1)
            _hdrs    = {'Authorization': f'token {_token}'}
            # POST /api/contents/<path>/checkpoints flushes current state to disk
            _sr = _req.post(f'{_base}/api/contents/{_rel}/checkpoints',
                            headers=_hdrs, timeout=15)
            if _sr.status_code in (200, 201):
                print(f'[HTML] Server save OK ({_sr.status_code}) — outputs flushed to disk')
                _saved = True
                time.sleep(2)
            # 403 is normal in SageMaker Studio (different auth layer) — silent fallback
    except Exception as _se:
        print(f'[HTML] Server save skipped: {_se}')
else:
    print('[HTML] requests not available — install with !pip install requests')

if not _saved:
    print('[HTML] Falling back to 5 s sleep for auto-save ...')
    time.sleep(5)

# -- Step 3: convert to HTML
_out_dir  = os.path.dirname(_nb_path)
_out_html = os.path.join(_out_dir, f'PS4_output_{_ts}.html')
_result = subprocess.run(
    ['jupyter', 'nbconvert', '--to', 'html', _nb_path, '--output', _out_html],
    capture_output=True, text=True
)
if _result.returncode == 0 and os.path.exists(_out_html):
    _size = os.path.getsize(_out_html) / 1024
    print(f'[HTML] Saved: {_out_html}  ({_size:.0f} KB)')
    print('  Open in browser — all cell outputs preserved permanently.')
else:
    print(f'[HTML] nbconvert failed (rc={_result.returncode}):')
    if _result.stderr: print(_result.stderr[-600:])
    # -- Step 4: re-run everything and capture fresh outputs
    print('[HTML] Retrying with --execute (re-runs all cells, ~15-30 min) ...')
    _out_html2 = os.path.join(_out_dir, f'PS4_executed_{_ts}.html')
    _r2 = subprocess.run(
        ['jupyter', 'nbconvert', '--to', 'html', '--execute',
         '--allow-errors', '--ExecutePreprocessor.timeout=7200',
         _nb_path, '--output', _out_html2],
        capture_output=True, text=True
    )
    if _r2.returncode == 0 and os.path.exists(_out_html2):
        _size2 = os.path.getsize(_out_html2) / 1024
        print(f'[HTML] Executed export saved: {_out_html2}  ({_size2:.0f} KB)')
    else:
        print('[HTML] Both attempts failed.')
        print(_r2.stderr[-500:] if _r2.stderr else '(no stderr)')
